# 🤖 Assistant statistique agentique
### Du RAG à l’agent — **version interactive** · Atelier STG17 · BAD / STATAFRIC · Jour 1

> **En une phrase :** vous allez construire, pièce par pièce, un assistant qui répond aux questions des utilisateurs à partir des publications d’un institut statistique — puis l’utiliser dans une **interface de conversation**, avec la trace de chaque décision, une file d’approbation humaine et un journal d’audit.

| ⏱ Durée | 🆓 Coût | 🔑 Clé d’API | 🇨🇮 Cas d’étude | 🧩 Dépendances |
|:--:|:--:|:--:|:--:|:--:|
| 90 min | 0 € | facultative | Côte d’Ivoire · données **fictives** | aucune librairie maison |

---

## 🧭 Comment utiliser ce notebook

1. **Dans Colab :** menu **Exécution → Tout exécuter** (ou `Ctrl + F9`). Comptez environ 30 secondes.
2. **Descendez jusqu’à l’étape 10** : l’interface de conversation apparaît. Posez vos questions.
3. **Remontez ensuite** pour comprendre chaque brique : chaque étape explique *pourquoi* elle existe.

> 💡 **Aucune clé n’est nécessaire.** Par défaut, l’interface utilise un « cerveau de démonstration » local, qui parle exactement le même langage qu’un vrai modèle. Pour un vrai modèle **gratuit**, ajoutez une clé Groq (étape 8).

---

## 🗺️ Le parcours

| Partie | Étape | Contenu | Ce que vous apprenez |
|:--|:--:|:--|:--|
| **🟦 Préparer** | 1 | Installation et affichage | Un environnement reproductible |
| | 2 | Le corpus | Travailler sur des sources maîtrisées |
| **🟩 Retrouver** | 3 | Deux moteurs de recherche, mesurés | Pourquoi l’approche B bat l’approche A |
| **🟨 Agir** | 4 | Les outils | Lecture, calcul, écriture : trois niveaux de risque |
| | 5 | Le protocole | Comment un modèle *demande* une action |
| | 6 | Politiques et file d’approbation | Où l’humain garde la main |
| | 7 | La boucle, l’ancrage, l’audit | Le cœur de l’agent et ses contrôles |
| **🟪 Penser** | 8 | Les « cerveaux » | Démo locale, Groq, Gemini ou Ollama |
| | 9 | Essai en mode texte | Vérifier avant d’ouvrir l’interface |
| **🟥 Utiliser** | 10 | **L’interface de conversation** | Un assistant utilisable par vos collègues |
| | 11 | Sous le capot | Ce qui se passe quand on clique sur « Envoyer » |
| **⬛ Conclure** | 12 | Limites, exercices, dépannage | Passer du laboratoire à la production |

---
# 🟦 PARTIE 1 — Préparer

## Étape 1 · Installation et outils d’affichage

**🎯 Objectif :** disposer d’un environnement identique pour tous les participants.

Cette cellule installe si nécessaire les bibliothèques standard (`scikit-learn`, `pandas`, `matplotlib`, `ipywidgets`) et définit la **charte d’affichage**. Tous les éléments visuels sont produits par du code Python : c’est ce qui garantit un rendu identique dans **Colab**, **Jupyter** et **VS Code**.

In [ ]:
# ── Étape 0 · Installation et configuration ─────────────────────────────────
import importlib, subprocess, sys

for module, paquet in [("sklearn", "scikit-learn>=1.3"), ("pandas", "pandas>=2.0"), ("matplotlib", "matplotlib>=3.7"), ("ipywidgets", "ipywidgets>=7.6"), ("requests", "requests")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquet], check=False)

import ast, hashlib, html, json, operator, os, re, shutil, time, unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Contexte du laboratoire
PAYS = {"iso3": "CIV", "nom": "Côte d’Ivoire"}
RACINE = Path("stg17_lab")
DOSSIER_CORPUS = RACINE / "corpus"
SORTIES = RACINE / "sorties" / PAYS["iso3"]
for dossier in (DOSSIER_CORPUS, SORTIES):
    dossier.mkdir(parents=True, exist_ok=True)

# Charte graphique
VERT, MARINE, OR, ROUGE, GRIS, CLAIR = "#1B7A43", "#0B2545", "#F2A900", "#B83B2E", "#6B7B75", "#EAF5EE"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9AA8A1", "axes.titleweight": "bold", "axes.titlesize": 12.5,
})

# ── Fonctions d’affichage ───────────────────────────────────────────────────
_STYLES = {
    "info":      (VERT,  "#EAF5EE", "💡"),
    "cle":       (MARINE, "#E8EEF6", "🔑"),
    "attention": (OR,    "#FFF7E0", "⚠️"),
    "danger":    (ROUGE, "#FBECEA", "⛔"),
    "succes":    (VERT,  "#E3F4E9", "✅"),
}

def encadre(texte, titre="", genre="info"):
    """Affiche un encadré coloré (info, cle, attention, danger, succes)."""
    couleur, fond, icone = _STYLES[genre]
    display(HTML(
        f"<div style='border-left:5px solid {couleur};background:{fond};padding:11px 16px;border-radius:8px;"
        f"margin:8px 0;font-family:Segoe UI,system-ui,sans-serif;color:#1d2b24;line-height:1.5'>"
        f"<b>{icone} {titre}</b><div style='margin-top:3px'>{texte}</div></div>"))

_CSS_TABLE = ("<style>.stg{border-collapse:collapse;font-family:Segoe UI,system-ui,sans-serif;font-size:12.5px;margin:6px 0}"
              ".stg th{background:#0B2545;color:#fff;padding:7px 10px;text-align:left}"
              ".stg td{padding:6px 10px;border-bottom:1px solid #DDE3E0;vertical-align:top}"
              ".stg tr:nth-child(even) td{background:#F4F7F5}</style>")

def tableau(df, legende=None, largeur_max=90):
    """Affiche un DataFrame avec la charte du laboratoire."""
    d = df.copy()
    for col in d.columns:
        if d[col].dtype == object:
            d[col] = d[col].map(lambda v: (str(v)[:largeur_max] + "…") if isinstance(v, str) and len(v) > largeur_max else v)
    bloc = _CSS_TABLE + d.to_html(index=False, classes="stg", border=0, escape=True)
    if legende:
        bloc += f"<div style='color:{GRIS};font-size:12px;font-style:italic;margin:2px 0 10px'>{legende}</div>"
    display(HTML(bloc))

def bandeau(surtitre, titre, sous_titre, couleur=MARINE):
    """Bandeau de section, rendu en HTML depuis Python (fiable dans Colab)."""
    display(HTML(
        f"<div style='background:linear-gradient(135deg,{couleur} 0%,#1B7A43 100%);border-radius:16px;padding:22px 30px;"
        f"font-family:Segoe UI,system-ui,sans-serif;margin:6px 0'>"
        f"<div style='color:#F2A900;font-size:12px;letter-spacing:3px;font-weight:700'>{surtitre}</div>"
        f"<div style='color:#fff;font-size:26px;font-weight:800;margin:6px 0'>{titre}</div>"
        f"<div style='color:#dbe7e0;font-size:14.5px;line-height:1.5'>{sous_titre}</div></div>"))

bandeau("ATELIER STG17 · BAD / STATAFRIC · JOUR 1", "🤖 Assistant statistique agentique",
        "Retrouver la bonne information · agir sous contrôle · tout tracer — "
        "<b style='color:#fff'>le modèle demande, votre code exécute.</b>")

encadre(f"Dossier de travail : <code>{RACINE}/</code><br>Sorties : <code>{SORTIES}</code>",
        "Environnement prêt", "succes")

## Étape 2 · Le corpus d’étude

**🎯 Objectif :** disposer de sources maîtrisées, dont on connaît toutes les réponses.

Neuf publications **fictives** d’un institut national de la statistique : emploi, prix, comptes nationaux, recensement, éducation, cacao, santé, pauvreté, et une note méthodologique.

> ⚠️ **Tous les chiffres sont inventés.** Ils ressemblent à de vraies statistiques, mais ne doivent jamais être cités.

**🔍 À observer :** le corpus est volontairement piégeux. Plusieurs documents parlent de « taux » et de « chômage », et certaines phrases ne nomment pas leur sujet (« Il s’établit à… »).

In [ ]:
# ── Le corpus : neuf publications fictives ──────────────────────────────────
CORPUS = {
"emploi_2023T4": ("Enquête nationale sur l’emploi — 4e trimestre 2023", """
Au quatrième trimestre 2023, le taux de chômage au sens du BIT s’établit à 8,6 % de la population active.
Le chômage touche davantage les jeunes de 15 à 24 ans. Pour cette tranche d’âge, il atteint 19,2 %, soit plus du double de la moyenne nationale.
Les femmes restent plus exposées que les hommes. Leur taux de chômage est de 10,1 %, contre 7,4 % pour les hommes.
Le taux d’activité de la population en âge de travailler se situe à 61,3 %.
L’emploi informel demeure prépondérant. Il représente 88,5 % de l’emploi total, avec une part plus élevée en milieu rural.
"""),
"ipc_2024_03": ("Indice harmonisé des prix à la consommation — mars 2024", """
En mars 2024, l’inflation en glissement annuel ressort à 4,1 %, après 4,5 % en février.
La hausse des prix est principalement tirée par les produits alimentaires et boissons non alcoolisées, dont les prix progressent de 6,3 % sur un an.
Les prix du transport augmentent de 2,2 % et ceux du logement, de l’eau et de l’électricité de 3,0 %.
L’inflation sous-jacente, qui exclut les produits frais et l’énergie, s’établit à 3,4 %.
"""),
"comptes_2023": ("Comptes nationaux annuels — 2023", """
L’économie a enregistré une croissance du produit intérieur brut réel de 6,5 % en 2023.
Le secteur primaire contribue pour 17 % à la valeur ajoutée, l’industrie pour 23 % et les services pour 50 %.
Il a été porté par les services financiers et les télécommunications, qui progressent de 9,8 %.
L’investissement public a représenté 7,2 % du PIB.
"""),
"rgph_2021": ("Recensement général de la population et de l’habitat — 2021", """
Le recensement dénombre 29,4 millions d’habitants résidant sur le territoire national.
La population a augmenté à un rythme annuel moyen de 2,9 % depuis le précédent recensement.
Plus de la moitié des habitants vit désormais en ville. Le taux d’urbanisation atteint 52,5 %.
La population est très jeune. Son âge médian est de 18,9 ans.
"""),
"education_2022": ("Annuaire statistique de l’éducation — 2022", """
Le taux brut de scolarisation au primaire est de 101 %, traduisant la présence d’élèves en dehors de l’âge officiel.
Le taux d’achèvement du cycle primaire s’élève à 81 %.
Chez les adultes de 15 ans et plus, le taux d’alphabétisation est estimé à 53 %.
Il y a en moyenne 41 élèves par enseignant dans les écoles primaires publiques.
"""),
"cacao_2023": ("Note de conjoncture — filière cacao, campagne 2022-2023", """
La production nationale de fèves de cacao est estimée à 2,2 millions de tonnes pour la campagne 2022-2023.
La filière représente environ 40 % des recettes d’exportation du pays.
Le prix bord champ garanti aux producteurs a été fixé à 1 000 FCFA par kilogramme.
"""),
"sante_2022": ("Tableau de bord de la santé — 2022", """
L’espérance de vie à la naissance est estimée à 59 ans.
Le taux de mortalité des enfants de moins de cinq ans s’élève à 75 décès pour 1 000 naissances vivantes.
La couverture vaccinale DTC3 des enfants de 12 à 23 mois atteint 84 %.
"""),
"pauvrete_2021": ("Profil de pauvreté — 2021", """
La proportion de la population vivant sous le seuil national de pauvreté est de 37,5 %.
En milieu rural, elle atteint 45 %, contre 25 % en milieu urbain.
Les inégalités de consommation, mesurées par l’indice de Gini, s’établissent à 0,35.
"""),
"note_methodo_emploi": ("Note méthodologique — définitions de l’enquête emploi", """
Un jeune est défini comme une personne âgée de 15 à 24 ans. Le chômage des jeunes est mesuré selon les critères du Bureau international du Travail.
Est considérée comme chômeuse toute personne sans emploi, disponible et ayant recherché activement un emploi.
L’emploi informel regroupe les emplois non déclarés et les unités de production non enregistrées.
Les taux présentés dans les publications trimestrielles sont pondérés et corrigés de la non-réponse.
"""),
}

for doc_id, (titre, texte) in CORPUS.items():
    (DOSSIER_CORPUS / f"{doc_id}.md").write_text(f"# {titre}\n{texte.strip()}\n", encoding="utf-8")

def charger_corpus(dossier):
    """Lit chaque fichier : 1re ligne = titre, reste = texte."""
    documents = []
    for chemin in sorted(Path(dossier).glob("*.md")):
        lignes = chemin.read_text(encoding="utf-8").splitlines()
        documents.append({"id": chemin.stem, "titre": lignes[0].lstrip("# ").strip(),
                          "texte": "\n".join(lignes[1:]).strip()})
    return documents

DOCS = charger_corpus(DOSSIER_CORPUS)
tableau(pd.DataFrame([{"identifiant": d["id"], "titre": d["titre"], "mots": len(d["texte"].split())} for d in DOCS]),
        f"{len(DOCS)} documents chargés depuis {DOSSIER_CORPUS} — données fictives.")

---
# 🟩 PARTIE 2 — Retrouver l’information

## Étape 3 · Deux moteurs de recherche, mesurés

**🎯 Objectif :** comprendre qu’un assistant ne vaut que par la qualité des passages qu’il retrouve.

Un **RAG** (*Retrieval-Augmented Generation*) retrouve d’abord les passages pertinents, puis répond **uniquement** à partir d’eux. Nous comparons deux moteurs :

| | 🅰️ Approche A — naïve | 🅱️ Approche B — améliorée |
|:--|:--|:--|
| **Découpage** | tous les 220 caractères, même au milieu d’un mot | par phrases entières, avec chevauchement |
| **Contexte** | aucun | le titre du document est ajouté à chaque passage |
| **Comparaison** | mots exacts (« chomage » ≠ « chômage ») | sans accents + fragments de 3 à 5 lettres |
| **Classement** | TF-IDF seul | BM25 **et** fragments, fusionnés par rangs (RRF) |

**📏 Comment on mesure :** 13 questions dont on connaît la réponse. Un passage est jugé **pertinent** s’il contient à la fois le chiffre attendu et un mot-indice.

- **Hit@1** — part des questions dont le 1er passage est pertinent ;
- **Hit@3** — part des questions avec un passage pertinent dans les 3 premiers ;
- **MRR** — moyenne de 1 / rang du premier passage pertinent (1 = parfait).

> 💡 Dans l’interface (étape 10), vous pourrez **basculer d’un moteur à l’autre** et constater la différence sur vos propres questions.

In [ ]:
# ── Étape 3 · Les deux moteurs de recherche ─────────────────────────────────
# 🅰️ Approche A : découpage fixe + TF-IDF sur les mots
def decouper_fixe(documents, taille=220):
    """Coupe chaque texte tous les `taille` caractères, sans chevauchement."""
    morceaux = []
    for doc in documents:
        texte = doc["texte"].replace("\n", " ")
        for debut in range(0, len(texte), taille):
            extrait = texte[debut:debut + taille]
            morceaux.append({"doc": doc["id"], "titre": doc["titre"], "texte": extrait, "contexte": extrait})
    return morceaux


class RechercheNaive:
    """Approche A — TF-IDF sur les mots, texte tel quel."""
    nom = "A · Naïf"

    def __init__(self, morceaux):
        self.morceaux = morceaux
        self.vectoriseur = TfidfVectorizer()
        self.matrice = self.vectoriseur.fit_transform([m["texte"] for m in morceaux])

    def chercher(self, question, k=3):
        scores = cosine_similarity(self.vectoriseur.transform([question]), self.matrice).ravel()
        ordre = np.argsort(-scores)[:k]
        return [{"morceau": self.morceaux[i], "score": float(scores[i]), "confiance": float(scores[i])} for i in ordre]


MORCEAUX_A = decouper_fixe(DOCS)
MOTEUR_A = RechercheNaive(MORCEAUX_A)


# 📏 Jeu d’évaluation et métriques
# (question, chiffre attendu, indice qui doit figurer dans le passage)
EVALUATION = [
    ("Quel est le taux de chomage des jeunes ?",                       "19,2",   "15 à 24"),
    ("À combien s’élève l’inflation annuelle en mars 2024 ?",          "4,1",    "mars 2024"),
    ("De combien ont augmenté les prix alimentaires ?",                "6,3",    "alimentaires"),
    ("Quelle est la croissance économique en 2023 ?",                  "6,5",    "croissance"),
    ("Combien d’habitants compte le pays ?",                           "29,4",   "habitants"),
    ("Quelle proportion de la population vit en ville ?",              "52,5",   "urbanisation"),
    ("Quelle part des emplois relève de l’informel ?",                 "88,5",   "informel"),
    ("Quel est le taux d’achèvement de l’école primaire ?",            "81",     "achèvement"),
    ("Quelle quantité de cacao a été produite ?",                      "2,2",    "cacao"),
    ("Quelle est l’esperance de vie a la naissance ?",                 "59",     "espérance"),
    ("Quel est le niveau de pauvreté dans les zones rurales ?",        "45",     "rural"),
    ("Quelle est la couverture vaccinale des enfants ?",               "84",     "vaccinale"),
    ("Le chômage des femmes est-il plus élevé que celui des hommes ?", "10,1",   "femmes"),
]

def _norm(texte):
    """Minuscules, sans accents : pour comparer indépendamment de la graphie."""
    texte = unicodedata.normalize("NFKD", texte.lower())
    return "".join(c for c in texte if not unicodedata.combining(c))

def est_pertinent(morceau, attendu, indice):
    contexte = _norm(morceau["contexte"])
    return _norm(attendu) in contexte and _norm(indice) in contexte

def format_rang(r):
    return "❌ absent du top 3" if pd.isna(r) else f"✅ {int(r)}"

def evaluer(moteur, k=3):
    lignes = []
    for question, attendu, indice in EVALUATION:
        resultats = moteur.chercher(question, k=k)
        rang = next((r + 1 for r, res in enumerate(resultats) if est_pertinent(res["morceau"], attendu, indice)), None)
        lignes.append({"question": question, "attendu": attendu, "rang": rang})
    detail = pd.DataFrame(lignes)
    synthese = {
        "moteur": moteur.nom,
        "Hit@1": (detail["rang"] == 1).mean(),
        "Hit@3": detail["rang"].notna().mean(),
        "MRR": detail["rang"].map(lambda r: 0.0 if pd.isna(r) else 1 / r).mean(),
    }
    return synthese, detail


# 🅱️ Approche B : chaîne de recherche améliorée
MOTS_VIDES = set("""le la les l de des du d un une et en a au aux est sont quel quelle quels quelles combien que qu qui
dans pour par sur ce cet cette ces il elle se s son sa ses leur leurs ou plus y avec comme""".split())

def tokeniser(texte):
    """Normalise puis découpe en mots utiles (sans accents, sans mots vides, pluriel simple retiré)."""
    mots = re.findall(r"[a-z0-9]+", _norm(texte))
    return [m[:-1] if len(m) > 4 and m.endswith("s") else m for m in mots if m not in MOTS_VIDES]


# ① + ② Découpage par phrases, avec chevauchement et en-tête contextuel
def decouper_phrases(documents, max_caracteres=260, chevauchement=1):
    """Regroupe des phrases entières ; la dernière phrase d’un morceau ouvre le suivant."""
    morceaux = []
    for doc in documents:
        phrases = [p.strip() for p in re.split(r"(?<=[.!?])\s+", doc["texte"].replace("\n", " ")) if p.strip()]
        groupes, courant = [], []
        for phrase in phrases:
            if courant and len(" ".join(courant + [phrase])) > max_caracteres:
                groupes.append(courant)
                courant = courant[-chevauchement:] if chevauchement else []
            courant.append(phrase)
        if courant:
            groupes.append(courant)
        for groupe in groupes:
            texte = " ".join(groupe)
            morceaux.append({"doc": doc["id"], "titre": doc["titre"], "texte": texte,
                             "contexte": f"[{doc['titre']}] {texte}"})   # ② en-tête contextuel
    return morceaux


# ④ BM25, implémenté en quelques lignes
class BM25:
    def __init__(self, documents_tokenises, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = documents_tokenises, k1, b
        self.longueurs = np.array([len(d) for d in documents_tokenises], dtype=float)
        self.longueur_moy = self.longueurs.mean()
        n = len(documents_tokenises)
        frequence_doc = {}
        for d in documents_tokenises:
            for mot in set(d):
                frequence_doc[mot] = frequence_doc.get(mot, 0) + 1
        self.idf = {m: np.log(1 + (n - f + 0.5) / (f + 0.5)) for m, f in frequence_doc.items()}

    def scores(self, requete):
        resultat = np.zeros(len(self.docs))
        for i, doc in enumerate(self.docs):
            compte = {}
            for mot in doc:
                compte[mot] = compte.get(mot, 0) + 1
            for mot in requete:
                if mot in compte:
                    tf = compte[mot]
                    norme = self.k1 * (1 - self.b + self.b * self.longueurs[i] / self.longueur_moy)
                    resultat[i] += self.idf[mot] * tf * (self.k1 + 1) / (tf + norme)
        return resultat


class RechercheHybride:
    """Approche B — BM25 (mots) + TF-IDF n-grammes de caractères, fusionnés par RRF."""
    nom = "B · Amélioré"

    def __init__(self, morceaux, k_rrf=60):
        self.morceaux, self.k_rrf = morceaux, k_rrf
        textes = [m["contexte"] for m in morceaux]                      # ② l’en-tête est indexé
        self.bm25 = BM25([tokeniser(t) for t in textes])                # ④
        self.vectoriseur = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)  # ③
        self.matrice = self.vectoriseur.fit_transform([_norm(t) for t in textes])

    def chercher(self, question, k=3):
        s_bm25 = self.bm25.scores(tokeniser(question))
        s_ngram = cosine_similarity(self.vectoriseur.transform([_norm(question)]), self.matrice).ravel()
        rrf = np.zeros(len(self.morceaux))
        for scores in (s_bm25, s_ngram):                                # ④ fusion par rangs
            for rang, i in enumerate(np.argsort(-scores)):
                rrf[i] += 1 / (self.k_rrf + rang + 1)
        ordre = np.argsort(-rrf)[:k]
        return [{"morceau": self.morceaux[i], "score": float(rrf[i]), "confiance": float(s_ngram[i])} for i in ordre]


MORCEAUX_B = decouper_phrases(DOCS)
MOTEUR_B = RechercheHybride(MORCEAUX_B)


# 🧾 Réponse ancrée (utilisée par le mode « RAG simple » de l’interface)
SEUIL_COSINUS = 0.30      # similarité (n-grammes) jugée suffisante à elle seule
SEUIL_COUVERTURE = 0.60   # ou : part des mots de la question retrouvés dans le passage

CONSIGNE_RAG = """Tu es l’assistant documentaire d’un institut national de la statistique.
Règles impératives :
1. Réponds UNIQUEMENT à partir des extraits fournis.
2. Cite la source de chaque chiffre entre crochets, ex. [emploi_2023T4].
3. Si les extraits ne contiennent pas la réponse, réponds exactement : « Information absente du corpus. »
4. N’arrondis pas et ne recalcule pas les chiffres."""

def construire_prompt(question, resultats):
    extraits = "\n".join(f"[{r['morceau']['doc']}] {r['morceau']['contexte']}" for r in resultats)
    return f"EXTRAITS :\n{extraits}\n\nQUESTION : {question}"

def reponse_extractive(question, resultats):
    """Sans modèle : renvoie le meilleur passage, mot pour mot, avec sa source."""
    meilleur = resultats[0]["morceau"]
    return f"« {meilleur['texte']} » [{meilleur['doc']}]"

def repondre(question, moteur=None, generateur=None):
    moteur = moteur or MOTEUR_B
    resultats = moteur.chercher(question, k=3)
    if not passage_suffisant(question, resultats[0])[0]:
        return {"question": question, "confiance": resultats[0]["confiance"], "mode": "refus (seuil)",
                "réponse": "Information absente du corpus."}
    if generateur is None:
        return {"question": question, "confiance": resultats[0]["confiance"], "mode": "extractif",
                "réponse": reponse_extractive(question, resultats)}
    texte = generateur(CONSIGNE_RAG, [{"role": "user", "content": construire_prompt(question, resultats)}])
    return {"question": question, "confiance": resultats[0]["confiance"], "mode": "LLM", "réponse": texte}


# 🛑 Savoir dire « je ne sais pas » : un double critère
MOTS_CONSIGNE = {"prepare", "redige", "note", "ecri", "fai", "donne", "indique", "quel", "niveau", "montre", "combien"}

def couverture(question, morceau):
    """Part des mots utiles de la question présents dans le passage (comparaison sur 5 lettres)."""
    mots = [m for m in tokeniser(question) if m not in MOTS_CONSIGNE and len(m) > 2]
    if not mots:
        return 0.0
    prefixes = {m[:5] for m in tokeniser(morceau["contexte"])}
    return sum(m[:5] in prefixes for m in mots) / len(mots)

def passage_suffisant(question, resultat):
    """Vrai si le meilleur passage est assez proche OU couvre l’essentiel de la question."""
    cov = couverture(question, resultat["morceau"])
    return (resultat["confiance"] >= SEUIL_COSINUS or cov >= SEUIL_COUVERTURE), cov

MOTEURS = {"B · Amélioré": MOTEUR_B, "A · Naïf": MOTEUR_A}
encadre(f"Approche A : <b>{len(MORCEAUX_A)}</b> passages · Approche B : <b>{len(MORCEAUX_B)}</b> passages · "
        f"refus si cosinus < <b>{SEUIL_COSINUS}</b> et couverture < <b>{SEUIL_COUVERTURE}</b>", "Moteurs prêts", "succes")

In [ ]:
# ── Étape 3 (suite) · La comparaison, en un graphique ───────────────────────
SYNTH_A, DETAIL_A = evaluer(MOTEUR_A)
SYNTH_B, DETAIL_B = evaluer(MOTEUR_B)

fig, ax = plt.subplots(figsize=(8.8, 3.4))
x = np.arange(3)
for j, (s, couleur) in enumerate([(SYNTH_A, "#A9B5B0"), (SYNTH_B, VERT)]):
    valeurs = [s["Hit@1"], s["Hit@3"], s["MRR"]]
    barres = ax.bar(x + (j - 0.5) * 0.36, valeurs, 0.34, color=couleur, label=s["moteur"])
    for b, v in zip(barres, valeurs):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks(x, ["Hit@1", "Hit@3", "MRR"]); ax.set_ylim(0, 1.28); ax.set_yticks([0, .25, .5, .75, 1])
ax.set_title("Qualité de la recherche sur 13 questions (plus haut = mieux)")
ax.legend(frameon=False, loc="upper center", ncol=2)
plt.tight_layout(); plt.show()

comparaison = pd.DataFrame({"question": DETAIL_A["question"],
                            "🅰️ Naïf": DETAIL_A["rang"].map(format_rang),
                            "🅱️ Amélioré": DETAIL_B["rang"].map(format_rang)})
tableau(comparaison, "Rang du premier passage pertinent, question par question.")
encadre(f"MRR : <b>{SYNTH_A['MRR']:.2f} → {SYNTH_B['MRR']:.2f}</b> · Hit@1 : <b>{SYNTH_A['Hit@1']:.0%} → {SYNTH_B['Hit@1']:.0%}</b>. "
        "Aucun modèle d’IA supplémentaire : un meilleur découpage, une meilleure normalisation, une meilleure combinaison. "
        "Sur un si petit jeu de questions, ces écarts restent indicatifs.", "Ce qu’il faut retenir", "cle")

### 🛑 Savoir dire « je ne sais pas »

Un assistant digne de confiance doit **refuser** quand le corpus ne contient pas la réponse. Un seul seuil de similarité ne suffit pas : les questions courtes (« inflation ») obtiennent un score faible alors qu’elles sont légitimes, et certaines questions hors sujet (« Qui est le ministre de l’économie ? ») obtiennent un score élevé grâce à un mot commun.

On combine donc **deux critères** : le passage est jugé suffisant si

- sa **similarité** avec la question est élevée (cosinus ≥ 0,30), **ou**
- il **couvre** au moins 60 % des mots utiles de la question.

**🔍 À observer :** le tableau montre la décision sur des questions légitimes et hors corpus. Aucune règle n’est parfaite : repérez le cas qui passe à tort, et souvenez-vous qu’il reste deux lignes de défense (la consigne du modèle et la relecture humaine).

In [ ]:
# ── Étape 3 (suite) · Le refus, question par question ───────────────────────
essais_refus = [("Quelle est l’inflation en mars 2024 ?", "légitime"), ("inflation", "légitime"),
                ("Combien d’habitants ?", "légitime"), ("esperence de vie", "légitime"),
                ("Quel est le taux de change du dollar américain ?", "hors corpus"),
                ("Quel est le prix du pétrole ?", "hors corpus"), ("Qui est le ministre de l’économie ?", "hors corpus"),
                ("Quel est le taux de mortalité maternelle ?", "hors corpus")]
lignes = []
for q, nature in essais_refus:
    meilleur = MOTEUR_B.chercher(q)[0]
    ok, cov = passage_suffisant(q, meilleur)
    attendu = nature == "légitime"
    lignes.append({"question": q, "nature": nature, "cosinus": f"{meilleur['confiance']:.2f}", "couverture": f"{cov:.0%}",
                   "décision": "✅ répond" if ok else "🛑 refuse", "verdict": "👍" if ok == attendu else "⚠️ erreur"})
tableau(pd.DataFrame(lignes), "« mortalité maternelle » passe à tort : le corpus parle de mortalité infantile. "
        "D’où la règle 3 de la consigne et la relecture humaine.")

---
# 🟨 PARTIE 3 — Agir, sous contrôle

## Étape 4 · Les outils

**🎯 Objectif :** définir précisément ce que l’agent a le droit de *demander*.

| Outil | Type | Rôle | Risque |
|:--|:--:|:--|:--|
| `chercher_documents` | 👁️ lecture | interroge le moteur B | faible |
| `lister_documents` | 👁️ lecture | inventaire des publications | faible |
| `calculer` | 👁️ lecture | arithmétique **sûre** | faible — si le code est protégé |
| `enregistrer_note` | ✍️ **écriture** | crée un fichier | **réel** : soumis à approbation |

**🧠 Trois choix de conception à retenir :**

- `ecrit` est un **champ** de l’outil, pas un commentaire : la politique ne peut pas l’oublier.
- La calculatrice **analyse** l’expression au lieu de l’exécuter : un modèle ne peut pas lui faire lancer du code.
- La recherche **refuse d’elle-même** quand aucun passage n’est assez pertinent : l’outil dit « je ne sais pas » avant que le modèle n’improvise.

In [ ]:
# ── Étape 6 · Les outils ────────────────────────────────────────────────────
@dataclass
class Outil:
    nom: str
    description: str
    parametres: dict               # nom du paramètre -> description
    fonction: Callable
    ecrit: bool = False            # ← la propriété qui compte

    def executer(self, **arguments):
        inconnus = set(arguments) - set(self.parametres)
        if inconnus:
            raise TypeError(f"paramètre(s) inconnu(s) : {sorted(inconnus)} ; attendus : {sorted(self.parametres)}")
        return str(self.fonction(**arguments))


def outil_recherche(moteur):
    def chercher_documents(requete):
        resultats = moteur.chercher(str(requete), k=3)
        if not passage_suffisant(str(requete), resultats[0])[0]:
            return "AUCUN PASSAGE PERTINENT : l’information semble absente du corpus."
        return "\n".join(f"[{r['morceau']['doc']}] {r['morceau']['texte']}" for r in resultats)
    return Outil("chercher_documents", "Recherche des passages dans les publications de l’institut.",
                 {"requete": "mots-clés ou question"}, chercher_documents)


def outil_liste(documents):
    def lister_documents():
        lignes = [f"{d['id']} — {d['titre']}" for d in documents]
        return f"{len(lignes)} publications disponibles :\n" + "\n".join(lignes)
    return Outil("lister_documents", "Liste les publications disponibles.", {}, lister_documents)


# Calculatrice sûre : on analyse l’expression au lieu de l’exécuter (amélioration n°1 de la partie B)
_OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
               ast.Div: operator.truediv, ast.Pow: operator.pow}
_UNAIRES = {ast.USub: operator.neg, ast.UAdd: operator.pos}

def calcul_sur(expression):
    expression = str(expression).replace(",", ".").strip()
    if len(expression) > 80:
        raise ValueError("expression trop longue")
    def evaluer_noeud(n):
        if isinstance(n, ast.Expression):
            return evaluer_noeud(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPERATIONS:
            gauche, droite = evaluer_noeud(n.left), evaluer_noeud(n.right)
            if isinstance(n.op, ast.Pow) and abs(droite) > 10:
                raise ValueError("exposant trop grand")
            return _OPERATIONS[type(n.op)](gauche, droite)
        if isinstance(n, ast.UnaryOp) and type(n.op) in _UNAIRES:
            return _UNAIRES[type(n.op)](evaluer_noeud(n.operand))
        raise ValueError(f"élément interdit : {type(n).__name__}")
    return round(evaluer_noeud(ast.parse(expression, mode="eval")), 6)

def outil_calcul():
    def calculer(expression):
        try:
            return calcul_sur(expression)
        except (ValueError, SyntaxError, ZeroDivisionError) as erreur:
            return f"ERREUR : {erreur}"
    return Outil("calculer", "Effectue un calcul arithmétique (+ - * / **).", {"expression": "ex. 19.2 - 8.6"}, calculer)


# Note : le nom de fichier est assaini (amélioration n°2 : pas de sortie du dossier autorisé)
def outil_note(dossier):
    dossier = Path(dossier)
    def enregistrer_note(nom_fichier, texte):
        nom = Path(str(nom_fichier)).name
        if not re.fullmatch(r"[\w\-]{1,60}\.md", nom):
            return f"ERREUR : nom de fichier refusé ({nom_fichier!r})"
        dossier.mkdir(parents=True, exist_ok=True)
        (dossier / nom).write_text(str(texte), encoding="utf-8")
        return f"note enregistrée : {nom}"
    return Outil("enregistrer_note", "Enregistre une note de synthèse (fichier .md).",
                 {"nom_fichier": "ex. constat.md", "texte": "contenu de la note"}, enregistrer_note, ecrit=True)


DOSSIER_NOTES = SORTIES / "notes"
shutil.rmtree(DOSSIER_NOTES, ignore_errors=True)       # repartir d’un dossier vide à chaque exécution
def construire_outils(moteur):
    """Les quatre outils, branchés sur le moteur de recherche choisi."""
    return [outil_recherche(moteur), outil_liste(DOCS), outil_calcul(), outil_note(DOSSIER_NOTES)]

OUTILS = construire_outils(MOTEUR_B)
OUTILS_PAR_NOM = {o.nom: o for o in OUTILS}

tableau(pd.DataFrame([{"outil": o.nom, "paramètres": ", ".join(o.parametres) or "—",
                       "type": "✍️ écriture" if o.ecrit else "👁️ lecture",
                       "dans l’interface": "🕒 approbation humaine" if o.ecrit else "✅ exécuté",
                       "description": o.description} for o in OUTILS]),
        "Dans l’interface, toute écriture conforme passe par la file d’approbation humaine (étape 6).")

In [ ]:
# La garde, démontrée : seules les opérations arithmétiques passent
calc = OUTILS_PAR_NOM["calculer"]
essais = ["19.2 - 8.6", "19,2 / 8,6", "(88.5 - 50) * 2", "__import__('os').system('echo piraté')",
          "open('/etc/passwd').read()", "9 ** 999999"]
tableau(pd.DataFrame([{"expression reçue du modèle": e, "résultat": calc.executer(expression=e)} for e in essais]),
        "Les trois dernières tentatives sont refusées sans jamais être exécutées.")

## Étape 5 · Le protocole : comment un modèle « demande »

**🎯 Objectif :** comprendre qu’un modèle ne fait **rien** ; il produit du texte qui *décrit* une action.

À chaque tour, le modèle répond par **un seul objet JSON** :

```json
{"pensee": "je cherche le taux", "outil": "chercher_documents", "args": {"requete": "chômage des jeunes"}}
```

ou, pour conclure :

```json
{"pensee": "j'ai tout", "reponse": "Le taux est de 19,2 % [chercher_documents]."}
```

**🔍 À observer :** l’analyseur est **tolérant** avec ce que les modèles produisent réellement (JSON entouré de texte ou de balises), mais **strict** pour le reste. Une réponse illisible n’arrête pas l’agent : l’erreur est renvoyée au modèle pour qu’il se corrige.

In [ ]:
# ── Étape 7 · La consigne système et l’analyseur ────────────────────────────
def decrire_outils(outils):
    lignes = []
    for o in outils:
        params = ", ".join(f"{p}: {d}" for p, d in o.parametres.items()) or "aucun"
        lignes.append(f"- {o.nom}({params}) — {o.description}{' [ÉCRITURE : soumis à approbation]' if o.ecrit else ''}")
    return "\n".join(lignes)

CONSIGNE_AGENT = """Tu es un agent d’analyse pour un institut national de la statistique.
Outils disponibles :
<<OUTILS>>

Format : réponds à CHAQUE tour par UN SEUL objet JSON, sans autre texte.
- Pour appeler un outil : {"pensee": "...", "outil": "<nom>", "args": {...}}
- Pour conclure : {"pensee": "...", "reponse": "..."}
Règles :
1. Utilise chercher_documents avant d’affirmer quoi que ce soit sur les données.
2. Utilise calculer pour toute opération arithmétique.
3. Cite entre crochets l’outil d’où provient chaque chiffre, ex. [chercher_documents].
4. Si un outil est refusé, ne réessaie pas : explique-le dans ta réponse.
5. Ne cite JAMAIS un chiffre qu’un outil ne t’a pas renvoyé."""

CONSIGNE_RENDUE = CONSIGNE_AGENT.replace("<<OUTILS>>", decrire_outils(OUTILS))

print(CONSIGNE_RENDUE)

In [ ]:
class ErreurProtocole(Exception):
    """La réponse du modèle ne peut pas être interprétée comme une action."""

@dataclass
class Action:
    pensee: str = ""
    outil: Optional[str] = None
    args: dict = field(default_factory=dict)
    reponse: Optional[str] = None

    @property
    def est_finale(self):
        return self.reponse is not None

def extraire_json(texte):
    """Trouve le premier objet JSON équilibré, même entouré de texte ou de balises ```."""
    debut = texte.find("{")
    while debut != -1:
        profondeur, dans_chaine, echappe = 0, False, False
        for i in range(debut, len(texte)):
            c = texte[i]
            if dans_chaine:
                echappe = (c == "\\") and not echappe
                if c == '"' and not echappe:
                    dans_chaine = False
                continue
            if c == '"':
                dans_chaine = True
            elif c == "{":
                profondeur += 1
            elif c == "}":
                profondeur -= 1
                if profondeur == 0:
                    try:
                        return json.loads(texte[debut:i + 1])
                    except json.JSONDecodeError:
                        break
        debut = texte.find("{", debut + 1)
    raise ErreurProtocole("aucun objet JSON exploitable dans la réponse")

def analyser_action(texte):
    objet = extraire_json(texte)
    if "reponse" in objet:
        return Action(pensee=str(objet.get("pensee", "")), reponse=str(objet["reponse"]))
    if "outil" in objet:
        args = objet.get("args", {})
        if not isinstance(args, dict):
            raise ErreurProtocole("« args » doit être un objet JSON")
        return Action(pensee=str(objet.get("pensee", "")), outil=str(objet["outil"]), args=args)
    raise ErreurProtocole("l’objet JSON ne contient ni « outil » ni « reponse »")

# Tolérant avec ce que les modèles produisent réellement, strict pour le reste
EXEMPLES = [
    '{"pensee":"je cherche","outil":"chercher_documents","args":{"requete":"chômage"}}',
    '```json\n{"outil":"calculer","args":{"expression":"2+2"}}\n```',
    'Bien sûr ! {"reponse":"8,6 % [chercher_documents]"} N’hésitez pas si…',
    '{"outil":"calculer","args":"19.2-8.6"}',
    'Je vais maintenant chercher dans les documents.',
]
lignes = []
for texte in EXEMPLES:
    try:
        a = analyser_action(texte)
        lignes.append({"réponse brute du modèle": texte.replace("\n", " ⏎ "), "verdict": "✅ accepté",
                       "interprétation": f"réponse : {a.reponse}" if a.est_finale else f"outil : {a.outil} {a.args}"})
    except ErreurProtocole as e:
        lignes.append({"réponse brute du modèle": texte, "verdict": "❌ rejeté", "interprétation": str(e)})
tableau(pd.DataFrame(lignes), "Un rejet n’arrête pas l’agent : l’erreur est renvoyée au modèle pour qu’il se corrige.")

## Étape 6 · Politiques et file d’approbation

**🎯 Objectif :** placer le contrôle humain **au bon endroit**.

Entre la **demande** du modèle et l’**exécution** de l’outil, une fonction décide : la **politique**. Elle renvoie « oui » ou « non », **avec un motif** journalisé.

```python
autorise, motif = politique(outil, arguments)   # ← « l’intervalle »
if not autorise:
    journaliser(motif); continuer               # l’outil n’est jamais appelé
resultat = outil.executer(**arguments)
```

**⭐ Nouveauté de cette version : la file d’approbation.** Dans une interface, on ne peut pas bloquer l’agent en attendant qu’un humain réponde. Une écriture conforme n’est donc **ni exécutée, ni refusée** : elle est **mise en attente**. Un analyste l’approuve ou la rejette ensuite, depuis l’onglet ✅ *Approbations* de l’interface. Chaque décision est horodatée et chaînée dans un journal.

| Situation | Décision |
|:--|:--|
| Lecture ordinaire | ✅ exécutée immédiatement |
| Demande contenant un identifiant individuel | ⛔ refusée (secret statistique) |
| Écriture non conforme (nom de fichier, mention « fictif », longueur) | ⛔ refusée |
| Écriture conforme | 🕒 **mise en file d’approbation** |

In [ ]:
# ── Étape 8 · Les politiques d’approbation ──────────────────────────────────
def politique_lecture_seule(outil, args):
    return (False, "écriture refusée par défaut") if outil.ecrit else (True, "lecture autorisée")

def politique_tout_autoriser(outil, args):
    return True, "tout est autorisé (démonstration)"

def politique_interactive(outil, args):
    if not outil.ecrit:
        return True, "lecture autorisée"
    reponse = input(f"L’agent veut exécuter {outil.nom}({args}). Autoriser ? [o/N] ")
    return (True, "approuvé par un humain") if reponse.strip().lower() == "o" else (False, "refusé par un humain")

# ⭐ Amélioration : une politique métier qui examine le contenu de la demande
MOTIF_IDENTIFIANT = re.compile(r"\b(id[_\- ]?r[ée]pondant|num[ée]ro de m[ée]nage|nni|\d{9,})\b", re.IGNORECASE)
TAILLE_MAX_NOTE = 2000

def politique_institut(outil, args):
    texte_args = " ".join(str(v) for v in args.values())
    if MOTIF_IDENTIFIANT.search(texte_args):
        return False, "demande portant sur un identifiant individuel (secret statistique)"
    if not outil.ecrit:
        return True, "lecture autorisée"
    nom = str(args.get("nom_fichier", ""))
    if not re.fullmatch(r"[\w\-]{1,60}\.md", nom):
        return False, f"nom de fichier non conforme : {nom!r}"
    if len(str(args.get("texte", ""))) > TAILLE_MAX_NOTE:
        return False, "note trop longue"
    if "fictif" not in str(args.get("texte", "")).lower():
        return False, "une note issue du corpus fictif doit porter la mention « fictif »"
    return True, "écriture conforme à la politique de l’institut"

# Une politique est du code : elle se teste comme du code
cas = [
    ("chercher_documents", {"requete": "taux de chômage des jeunes"}),
    ("chercher_documents", {"requete": "revenu du ménage id_répondant 004512"}),
    ("enregistrer_note",   {"nom_fichier": "constat.md", "texte": "Écart de 10,6 points (corpus fictif)."}),
    ("enregistrer_note",   {"nom_fichier": "../../etc/passwd", "texte": "..."}),
    ("enregistrer_note",   {"nom_fichier": "constat.md", "texte": "Écart de 10,6 points."}),
]
lignes = []
for nom_outil, args in cas:
    ligne = {"outil": nom_outil, "arguments": json.dumps(args, ensure_ascii=False)}
    for politique in (politique_lecture_seule, politique_tout_autoriser, politique_institut):
        ok, motif = politique(OUTILS_PAR_NOM[nom_outil], args)
        ligne[politique.__name__.replace("politique_", "")] = f"{'✅' if ok else '⛔'} {motif}"
    lignes.append(ligne)
tableau(pd.DataFrame(lignes), "Même demande, trois politiques : seule la politique métier bloque la fuite de données "
        "et le chemin malveillant tout en autorisant la note conforme.", largeur_max=70)

In [ ]:
# ── Étape 6 (suite) · La file d’approbation humaine ─────────────────────────
@dataclass
class DemandeApprobation:
    numero: int
    outil: str
    args: dict
    tache: str
    creee_le: str
    statut: str = "en attente"            # en attente · approuvée · rejetée
    decideur: str = ""
    decidee_le: str = ""
    resultat: str = ""


class FileApprobation:
    """Les écritures conformes attendent une décision humaine ; chaque décision est chaînée."""

    def __init__(self, outils_par_nom):
        self.outils = outils_par_nom
        self.demandes, self.decisions = [], []
        self.tache_courante = ""

    def politique_file_approbation(self, outil, args):
        autorise, motif = politique_institut(outil, args)
        if not autorise or not outil.ecrit:
            return autorise, motif
        demande = DemandeApprobation(len(self.demandes) + 1, outil.nom, dict(args), self.tache_courante,
                                     datetime.now(timezone.utc).isoformat(timespec="seconds"))
        self.demandes.append(demande)
        return False, f"mise en file d’approbation (demande n°{demande.numero}) : rien n’est écrit avant validation humaine"

    def en_attente(self):
        return [d for d in self.demandes if d.statut == "en attente"]

    def decider(self, numero, approuver, decideur="analyste"):
        demande = next(d for d in self.demandes if d.numero == numero)
        if demande.statut != "en attente":
            return demande
        if approuver:
            demande.resultat = self.outils[demande.outil].executer(**demande.args)
            demande.statut = "approuvée"
        else:
            demande.statut = "rejetée"
        demande.decideur = decideur
        demande.decidee_le = datetime.now(timezone.utc).isoformat(timespec="seconds")
        precedent = self.decisions[-1]["hash"] if self.decisions else "0" * 64
        entree = {"numero": numero, "outil": demande.outil, "args": demande.args, "statut": demande.statut,
                  "decideur": decideur, "horodatage": demande.decidee_le, "hash_precedent": precedent}
        entree["hash"] = hashlib.sha256(json.dumps(entree, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        self.decisions.append(entree)
        return demande


FILE = FileApprobation(OUTILS_PAR_NOM)

# Démonstration : une note conforme est mise en attente, puis approuvée
note = OUTILS_PAR_NOM["enregistrer_note"]
ok, motif = FILE.politique_file_approbation(note, {"nom_fichier": "demo.md", "texte": "Test de la file (corpus fictif)."})
avant = DOSSIER_NOTES.exists() and any(DOSSIER_NOTES.iterdir())
FILE.decider(1, approuver=True)
apres = sorted(p.name for p in DOSSIER_NOTES.glob("*"))
tableau(pd.DataFrame([
    {"moment": "① le modèle demande l’écriture", "décision": f"{'✅' if ok else '🕒'} {motif}", "fichier sur le disque": "oui" if avant else "non"},
    {"moment": "② un analyste approuve", "décision": FILE.demandes[0].statut, "fichier sur le disque": ", ".join(apres)},
]), "Le fichier n’existe qu’après la décision humaine. On vérifie le disque, pas seulement le journal.")
(DOSSIER_NOTES / "demo.md").unlink(missing_ok=True)
FILE = FileApprobation(OUTILS_PAR_NOM)      # file remise à zéro pour l’interface

## Étape 7 · La boucle, l’ancrage et l’audit

**🎯 Objectif :** assembler le cœur de l’agent et ses deux contrôles de sortie.

**🔁 La boucle** répète trois gestes jusqu’à une réponse finale ou l’épuisement du budget (`max_etapes`) :

1. **Analyser** la demande du modèle (JSON valide ? outil connu ?) ;
2. **Consulter la politique** (autorisé ? refusé ? en attente ?) ;
3. **Exécuter et journaliser**, puis renvoyer le résultat au modèle.

**✅ Le contrôle d’ancrage** vérifie que chaque nombre de la réponse finale figure dans les résultats d’outils, et que chaque source citée a été réellement appelée. Il détecte les **chiffres inventés**.

**🧾 Le journal d’audit** conserve ce que le modèle a *réellement écrit*. Chaque entrée contient l’empreinte SHA-256 de la précédente : une modification après coup **casse la chaîne**.

In [ ]:
# ── Étape 9 · L’agent ───────────────────────────────────────────────────────
@dataclass
class Etape:
    numero: int
    brut: str                                  # ce que le modèle a réellement écrit
    action: Optional[Action] = None
    autorise: Optional[bool] = None
    motif: str = ""
    resultat: Optional[str] = None
    erreur: Optional[str] = None
    duree_ms: float = 0.0

@dataclass
class Execution:
    tache: str
    politique: str
    etapes: list = field(default_factory=list)
    reponse: str = ""
    arret: str = ""

    def journal(self):
        lignes = []
        for e in self.etapes:
            a = e.action
            lignes.append({
                "étape": e.numero,
                "demande": ("réponse finale" if a and a.est_finale else (a.outil if a else "— illisible —")),
                "arguments": json.dumps(a.args, ensure_ascii=False) if a and not a.est_finale else "",
                "décision": "" if e.autorise is None else ("✅ autorisé" if e.autorise else ("🕒 en attente" if e.motif.startswith("mise en file") else "⛔ refusé")),
                "motif / erreur": e.erreur or e.motif,
                "résultat": (e.resultat or "").replace("\n", " ")[:90],
                "ms": round(e.duree_ms, 1),
            })
        return pd.DataFrame(lignes)


class ModeleScripte:
    """Rejoue des réponses prédéfinies. Aucune décision, aucune clé d’API."""
    def __init__(self, reponses):
        self.reponses, self.appels = list(reponses), 0
    def __call__(self, consigne, messages):
        self.appels += 1
        return self.reponses.pop(0) if self.reponses else '{"reponse": "(fin du script)"}'


class Agent:
    def __init__(self, outils, modele, politique=politique_lecture_seule, max_etapes=8):
        self.outils = {o.nom: o for o in outils}
        self.modele, self.politique, self.max_etapes = modele, politique, max_etapes
        self.consigne = CONSIGNE_AGENT.replace("<<OUTILS>>", decrire_outils(outils))

    def executer(self, tache, bavard=True):
        run = Execution(tache=tache, politique=self.politique.__name__)
        messages = [{"role": "user", "content": tache}]
        afficher = print if bavard else (lambda *a, **k: None)
        afficher(f"🎯 TÂCHE : {tache}\n")

        for n in range(1, self.max_etapes + 1):
            t0 = time.perf_counter()
            brut = self.modele(self.consigne, messages)
            messages.append({"role": "assistant", "content": brut})
            etape = Etape(numero=n, brut=brut)
            run.etapes.append(etape)

            # ① Analyser
            try:
                action = analyser_action(brut)
                etape.action = action
            except ErreurProtocole as e:
                etape.erreur = f"protocole : {e}"
                messages.append({"role": "user", "content": f"ERREUR DE PROTOCOLE : {e}. Réponds uniquement par un objet JSON."})
                afficher(f"  {n}. ❌ réponse illisible → erreur renvoyée au modèle")
                continue

            if action.est_finale:
                run.reponse, run.arret = action.reponse, "réponse finale"
                afficher(f"  {n}. 🏁 réponse finale")
                break

            outil = self.outils.get(action.outil)
            if outil is None:
                etape.erreur = f"outil inconnu : {action.outil}"
                messages.append({"role": "user", "content": f"ERREUR : l’outil « {action.outil} » n’existe pas. "
                                 f"Outils valides : {', '.join(self.outils)}."})
                afficher(f"  {n}. ❓ outil inexistant « {action.outil} » → erreur renvoyée")
                continue

            # ② Politique : l’intervalle
            etape.autorise, etape.motif = self.politique(outil, action.args)
            if not etape.autorise:
                statut = "EN ATTENTE D’APPROBATION HUMAINE" if "file d’approbation" in etape.motif else "REFUSÉ"
                messages.append({"role": "user", "content": f"RÉSULTAT DE {outil.nom} : {statut} — {etape.motif}."})
                afficher(f"  {n}. {'🕒' if statut.startswith('EN ATTENTE') else '⛔'} {outil.nom} : {etape.motif}")
                continue

            # ③ Exécuter
            try:
                etape.resultat = outil.executer(**action.args)
            except TypeError as e:
                etape.erreur = f"arguments invalides : {e}"
            etape.duree_ms = (time.perf_counter() - t0) * 1000
            retour = etape.resultat if etape.erreur is None else f"ERREUR : {etape.erreur}"
            messages.append({"role": "user", "content": f"RÉSULTAT DE {outil.nom} :\n{retour}"})
            afficher(f"  {n}. 🛠️ {outil.nom}({json.dumps(action.args, ensure_ascii=False)[:60]}) → "
                     f"{retour.splitlines()[0][:70]}")
        else:
            run.arret = f"budget épuisé ({self.max_etapes} étapes)"
            run.reponse = "Arrêt : budget atteint sans réponse finale."
            afficher(f"  ⏹️ {run.arret}")

        afficher(f"\n💬 RÉPONSE : {run.reponse}")
        return run

# ✅ Contrôle d’ancrage
def _nombres(texte):
    return {float(n.replace(",", ".")) for n in re.findall(r"\d+(?:[.,]\d+)?", texte)}

def verifier_ancrage(run):
    observations = " ".join(e.resultat or "" for e in run.etapes)
    connus = _nombres(observations)
    outils_appeles = {e.action.outil for e in run.etapes if e.action and not e.action.est_finale and e.resultat}
    lignes = [{"élément": f"nombre {n:g}", "statut": "✅ ancré" if n in connus else "🚩 non trouvé dans les résultats d’outils"}
              for n in sorted(_nombres(run.reponse))]
    for bloc in re.findall(r"\[([^\]]+)\]", run.reponse):
        for source in (s.strip() for s in bloc.split(",")):
            lignes.append({"élément": f"source [{source}]",
                           "statut": "✅ outil réellement appelé" if source in outils_appeles else "🚩 source jamais appelée"})
    rapport = pd.DataFrame(lignes)
    return rapport, (not rapport.empty and rapport["statut"].str.startswith("✅").all())


# 🧾 Journal d’audit chaîné
def _empreinte(objet):
    return hashlib.sha256(json.dumps(objet, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def sauvegarder_journal(run, chemin, meta):
    precedent, entrees = "0" * 64, []
    for e in run.etapes:
        a = e.action
        entree = {
            "etape": e.numero, "brut": e.brut,
            "outil": (a.outil if a and not a.est_finale else None), "args": (a.args if a and not a.est_finale else None),
            "finale": bool(a and a.est_finale), "autorise": e.autorise, "motif": e.motif,
            "resultat": e.resultat, "erreur": e.erreur, "hash_precedent": precedent,
        }
        entree["hash"] = precedent = _empreinte(entree)
        entrees.append(entree)
    rapport, ancre = verifier_ancrage(run)
    document = {
        "meta": {**meta, "horodatage": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                 "tache": run.tache, "politique": run.politique, "arret": run.arret,
                 "reponse": run.reponse, "ancrage_complet": bool(ancre)},
        "etapes": entrees,
        "sceau": precedent,
    }
    Path(chemin).write_text(json.dumps(document, ensure_ascii=False, indent=2), encoding="utf-8")
    return chemin

def verifier_journal(chemin):
    document = json.loads(Path(chemin).read_text(encoding="utf-8"))
    precedent = "0" * 64
    for entree in document["etapes"]:
        copie = {k: v for k, v in entree.items() if k != "hash"}
        if copie["hash_precedent"] != precedent or _empreinte(copie) != entree["hash"]:
            return False, f"chaîne rompue à l’étape {entree['etape']}"
        precedent = entree["hash"]
    return (precedent == document["sceau"]), ("intègre" if precedent == document["sceau"] else "sceau invalide")

encadre("Boucle, contrôle d’ancrage et journal chaîné sont prêts.", "Cœur de l’agent assemblé", "succes")

---
# 🟪 PARTIE 4 — Les « cerveaux »

## Étape 8 · Démo locale ou modèle gratuit

**🎯 Objectif :** comprendre que le cerveau est **interchangeable**. L’agent, les outils, la politique et le journal ne changent pas.

| Cerveau | Clé | Quota | Quand l’utiliser |
|:--|:--:|:--|:--|
| 🧪 **Démo locale** (règles) | aucune | illimité | Formation, tests, démonstration hors ligne |
| ⚡ **Groq** · `llama-3.1-8b-instant` | `GROQ_API_KEY` | ≈ 14 400 requêtes/jour* | Le meilleur quota gratuit, très rapide |
| 🔷 **Gemini** · `gemini-flash-lite-latest` | `GEMINI_API_KEY` | quelques centaines/jour* | Modèle plus capable, contexte long |
| 🖥️ **Ollama** · `qwen2.5:3b` | aucune | illimité | Données sensibles : rien ne quitte la machine |

*\* Quotas publiés en 2026, susceptibles d’évoluer : vérifiez la page « Limits » de votre compte.*

**🔑 Ajouter une clé dans Colab :** icône **🔑 Secrets** dans la barre de gauche → **Ajouter un secret** → nom `GROQ_API_KEY` → collez la clé → activez **Accès au notebook**. Clé gratuite sur `console.groq.com`.

**🧪 Le cerveau de démonstration** applique des règles simples (chercher, calculer si l’on demande un écart, préparer une note si l’on en demande une), mais il répond **dans le même format JSON** qu’un vrai modèle. Toute la chaîne de contrôle est donc réellement exercée.

> ⚠️ **Gratuit ne veut pas dire sans conditions.** Avant d’envoyer de vraies données d’un institut à un service en ligne, lisez ses conditions d’utilisation. Pour des données confidentielles, préférez Ollama.

In [ ]:
# ── Étape 8 · Le cerveau de démonstration (aucun réseau, même protocole) ────
class ModeleRegles:
    """Planificateur à règles : il parle le protocole JSON exactement comme un LLM."""
    fournisseur, modele = "démo", "règles locales"

    def __init__(self):
        self.appels, self.reprises, self.jetons = 0, 0, 0

    def __repr__(self):
        return "🧪 démo · règles locales"

    @staticmethod
    def _repondre(pensee, **action):
        return json.dumps({"pensee": pensee, **action}, ensure_ascii=False)

    def __call__(self, consigne, messages):
        self.appels += 1
        tache = messages[0]["content"]
        t = _norm(tache)
        retours = [m["content"] for m in messages[1:] if m["role"] == "user"]
        def resultat(nom):
            return next((r.split("\n", 1)[1] if "\n" in r else r for r in retours if r.startswith(f"RÉSULTAT DE {nom}")), None)

        veut_liste = any(k in t for k in ("quels documents", "liste", "publications disponibles", "combien de documents", "combien de publications"))
        veut_ecart = any(k in t for k in ("ecart", "difference", "depasse", "compar", "plus eleve", "moins eleve"))
        veut_note = any(k in t for k in ("note", "enregistr", "sauvegard", "memo"))
        sujet = [m for m in tokeniser(tache) if m not in ("document", "publication", "disponible", "liste", "note", "enregistre")]

        # ① Inventaire demandé
        if veut_liste and resultat("lister_documents") is None:
            return self._repondre("On me demande l’inventaire des publications.", outil="lister_documents", args={})
        # ② Recherche, dès qu’il y a un sujet
        if sujet and resultat("chercher_documents") is None:
            requete = " ".join(m for m in sujet if m not in MOTS_CONSIGNE) if veut_note else tache
            return self._repondre("Je cherche les passages pertinents.", outil="chercher_documents", args={"requete": requete or tache})
        passage = resultat("chercher_documents") or ""
        refus = passage.startswith("RÉSULTAT DE") or passage.startswith("ERREUR")
        if refus:
            motif = passage.split("—", 1)[-1].strip() if "—" in passage else passage
            return self._repondre("Ma recherche a été bloquée : je l’explique.",
                                  reponse=f"⛔ Je ne peux pas traiter cette demande : {motif}")
        trouve = bool(passage) and not passage.startswith("AUCUN PASSAGE")
        # ③ Calcul d’un écart entre les deux premiers pourcentages du meilleur passage
        pourcentages = re.findall(r"(\d+(?:,\d+)?)\s?%", passage.split("\n")[0]) if trouve else []
        if veut_ecart and len(pourcentages) >= 2 and resultat("calculer") is None:
            a, b = sorted(float(p.replace(",", ".")) for p in pourcentages[:2])
            return self._repondre("Je calcule l’écart avec l’outil, sans l’estimer.", outil="calculer", args={"expression": f"{b} - {a}"})
        # ④ Note demandée : préparée, puis soumise à la politique
        if veut_note and trouve and resultat("enregistrer_note") is None:
            nom = "note_" + "_".join(sujet[:3])[:40] + ".md"
            elements = "\n".join(f"- {l}" for l in passage.split("\n")[:2])
            texte = f"Question : {tache}\nÉléments retenus :\n{elements}\n(Source : corpus fictif de l’atelier STG17)"
            return self._repondre("Je prépare la note demandée.", outil="enregistrer_note", args={"nom_fichier": nom, "texte": texte})

        # ⑤ Réponse finale, construite uniquement à partir des résultats d’outils
        morceaux = []
        inventaire = resultat("lister_documents")
        if inventaire:
            premiere = inventaire.split("\n")[0].rstrip(" :")
            morceaux.append(f"**{premiere}** [lister_documents].")
        if passage and veut_note and trouve:
            extraits = []
            for l in passage.split("\n")[:2]:
                doc, _, texte = l.partition("] ")
                extraits.append(f"- « {texte} » (`{doc.lstrip('[')}`)")
            morceaux.append("Éléments retenus pour la note [chercher_documents] :\n" + "\n".join(extraits))
        elif passage:
            if trouve:
                doc, _, texte = passage.split("\n")[0].partition("] ")
                morceaux.append(f"D’après la publication `{doc.lstrip('[')}` [chercher_documents] :\n« {texte} »")
            else:
                morceaux.append("Je n’ai trouvé **aucune information fiable** sur ce sujet dans le corpus [chercher_documents].")
        calcul = resultat("calculer")
        if calcul and not calcul.startswith("ERREUR"):
            morceaux.append(f"➡️ L’écart est de **{calcul.replace('.', ',')} points de pourcentage** [calculer].")
        ecriture = resultat("enregistrer_note")
        if ecriture:
            if "EN ATTENTE" in ecriture:
                morceaux.append("📝 La note a été préparée et placée en **file d’approbation** : elle ne sera écrite qu’après validation par un analyste.")
            else:
                morceaux.append(f"⛔ La note n’a pas été enregistrée : {ecriture.split('—', 1)[-1].strip()}")
        if not morceaux:
            morceaux.append("Pouvez-vous préciser votre question ? Je peux chercher dans les publications, lister les documents, "
                            "calculer un écart ou préparer une note.")
        return self._repondre("J’ai les éléments nécessaires.", reponse="\n".join(morceaux))


encadre("Le cerveau de démonstration est prêt : il ne consulte aucun service en ligne.", "Cerveau local", "succes")

In [ ]:
# ── Étape 8 (suite) · Connecteur vers les modèles gratuits ─────────────────
FOURNISSEUR = "auto"      # "auto", "groq", "gemini" ou "ollama"
MODELE = None             # None = modèle par défaut du fournisseur ; ex. "openai/gpt-oss-120b"

FOURNISSEURS = {
    "groq":   {"url": "https://api.groq.com/openai/v1", "cle": "GROQ_API_KEY",
               "modele": "llama-3.1-8b-instant", "appels_par_minute": 30},
    "gemini": {"url": "https://generativelanguage.googleapis.com/v1beta/openai", "cle": "GEMINI_API_KEY",
               "modele": "gemini-flash-lite-latest", "appels_par_minute": 10},
    "ollama": {"url": os.environ.get("OLLAMA_URL", "http://localhost:11434") + "/v1", "cle": None,
               "modele": "qwen2.5:3b", "appels_par_minute": 600},
}

def lire_secret(nom):
    """Cherche une clé dans les variables d'environnement, puis dans les Secrets de Colab."""
    if not nom:
        return None
    valeur = os.environ.get(nom)
    if not valeur:
        try:
            from google.colab import userdata
            valeur = userdata.get(nom)
        except Exception:
            valeur = None
    return valeur

def ollama_disponible():
    import requests
    try:
        return requests.get(FOURNISSEURS["ollama"]["url"].removesuffix("/v1") + "/api/tags", timeout=2).ok
    except Exception:
        return False


class ErreurFournisseur(Exception):
    pass


class ModeleCompatibleOpenAI:
    """Adaptateur (consigne, messages) -> texte pour toute API /chat/completions."""

    def __init__(self, fournisseur="auto", modele=None, temperature=0.0, max_tokens=700, max_reprises=4):
        self.fournisseur = self._choisir(fournisseur)
        conf = FOURNISSEURS[self.fournisseur]
        self.url = conf["url"].rstrip("/") + "/chat/completions"
        self.cle = lire_secret(conf["cle"]) if conf["cle"] else "ollama"
        self.modele = modele or conf["modele"]
        self.intervalle = 60.0 / conf["appels_par_minute"] * 1.05      # marge de 5 %
        self.temperature, self.max_tokens, self.max_reprises = temperature, max_tokens, max_reprises
        self.appels, self.reprises, self.jetons, self._dernier = 0, 0, 0, 0.0

    @staticmethod
    def _choisir(fournisseur):
        if fournisseur != "auto":
            conf = FOURNISSEURS[fournisseur]
            if conf["cle"] and not lire_secret(conf["cle"]):
                raise ErreurFournisseur(f"clé {conf['cle']} absente")
            if fournisseur == "ollama" and not ollama_disponible():
                raise ErreurFournisseur("serveur Ollama injoignable")
            return fournisseur
        for nom in ("groq", "gemini"):
            if lire_secret(FOURNISSEURS[nom]["cle"]):
                return nom
        if ollama_disponible():
            return "ollama"
        raise ErreurFournisseur("aucun fournisseur : ajoutez GROQ_API_KEY ou GEMINI_API_KEY, ou lancez Ollama")

    def __call__(self, consigne, messages):
        import requests
        corps = {"model": self.modele, "temperature": self.temperature, "max_tokens": self.max_tokens,
                 "messages": [{"role": "system", "content": consigne}] + messages}
        for tentative in range(self.max_reprises + 1):
            attente = self.intervalle - (time.monotonic() - self._dernier)      # limiteur de débit
            if attente > 0:
                time.sleep(attente)
            self._dernier = time.monotonic()
            self.appels += 1
            r = requests.post(self.url, json=corps, timeout=120,
                              headers={"Authorization": f"Bearer {self.cle}", "Content-Type": "application/json"})
            if r.status_code in (429, 500, 502, 503) and tentative < self.max_reprises:
                try:
                    delai = float(r.headers.get("retry-after") or 0)
                except ValueError:
                    delai = 0
                delai = min(90, delai or 2 ** (tentative + 1))
                self.reprises += 1
                motif = "quota atteint" if r.status_code == 429 else "serveur indisponible"
                print(f"     ⏳ HTTP {r.status_code} ({motif}) → nouvel essai dans {delai:.0f} s")
                time.sleep(delai)
                continue
            if not r.ok:
                raise ErreurFournisseur(f"HTTP {r.status_code} : {r.text[:200]}")
            donnees = r.json()
            self.jetons += (donnees.get("usage") or {}).get("total_tokens", 0) or 0
            return donnees["choices"][0]["message"].get("content") or ""
        raise ErreurFournisseur("quota toujours atteint après plusieurs essais")

    def __repr__(self):
        return f"{self.fournisseur} · {self.modele}"


tableau(pd.DataFrame([{"fournisseur": n, "variable de clé": c["cle"] or "— (local)",
                       "disponible": "✅" if (lire_secret(c["cle"]) if c["cle"] else ollama_disponible()) else "—",
                       "modèle par défaut": c["modele"], "appels/min visés": c["appels_par_minute"]}
                      for n, c in FOURNISSEURS.items()]),
        "Détection automatique des fournisseurs disponibles.")

In [ ]:
# ── Optionnel · Installer Ollama sur Colab (modèle local, illimité, sans clé) ──
INSTALLER_OLLAMA = False   # passez à True, puis exécutez la cellule (≈ 2 à 4 minutes)

if INSTALLER_OLLAMA:
    if not shutil.which("ollama"):
        subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=False)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", FOURNISSEURS["ollama"]["modele"]], check=False)
    encadre(f"Ollama disponible : <b>{ollama_disponible()}</b>. Exécutez maintenant la cellule suivante. "
            "Sur Colab, un environnement GPU (Exécution → Modifier le type d’exécution) accélère nettement les réponses.",
            "Ollama", "info")
else:
    encadre("Ollama n’est pas installé. Passez <code>INSTALLER_OLLAMA = True</code> pour un modèle local, illimité et "
            "confidentiel — sans aucune clé.", "Option locale", "info")

## Étape 9 · Essai en mode texte

**🎯 Objectif :** vérifier que tout fonctionne **avant** d’ouvrir l’interface.

La fonction `demander()` produit exactement le même affichage que l’interface : utile dans les environnements où les widgets interactifs ne s’affichent pas (aperçu GitHub, certains éditeurs).

```python
demander("Quel est le taux de chômage des jeunes ?")
demander("Quelle est l’inflation en mars 2024 ?", mode="rag", moteur="A · Naïf")
```

In [ ]:
# ── Étape 9 · Mise en forme des réponses et mode texte ──────────────────────
ICONES_SOURCES = {"chercher_documents": "🔎", "calculer": "🧮", "lister_documents": "📚", "enregistrer_note": "📝"}

def _pastille(source):
    icone = ICONES_SOURCES.get(source, "📄")
    return (f"<span style='display:inline-block;background:#E8F5EF;color:#0F5A31;border:1px solid #BFE3CE;"
            f"border-radius:12px;padding:0 8px;margin:0 2px;font-size:11.5px;white-space:nowrap'>{icone} {source}</span>")

def texte_vers_html(texte):
    """Convertit le texte du modèle (Markdown léger) en HTML sûr et lisible."""
    lignes_html, liste = [], []
    def vider_liste():
        if liste:
            lignes_html.append("<ul style='margin:4px 0 4px 18px;padding:0'>" + "".join(f"<li>{l}</li>" for l in liste) + "</ul>")
            liste.clear()
    for brute in str(texte).strip().splitlines():
        ligne = html.escape(brute.strip())
        ligne = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", ligne)
        ligne = re.sub(r"(?<![\w*])\*(?!\s)(.+?)\*(?!\w)", r"<i>\1</i>", ligne)
        ligne = re.sub(r"`([^`]+)`", r"<code style='background:#EEF2F0;color:#0B2545;padding:1px 5px;border-radius:4px'>\1</code>", ligne)
        ligne = re.sub(r"\[([A-Za-z0-9_ ,\-]{3,80})\]", lambda m: "".join(_pastille(s.strip()) for s in m.group(1).split(",")), ligne)
        if re.match(r"^(-|\*|•)\s+", brute.strip()):
            liste.append(re.sub(r"^(-|\*|•)\s+", "", ligne))
            continue
        vider_liste()
        if brute.strip().startswith("#"):
            ligne = f"<b style='color:#0B2545'>{ligne.lstrip('#').strip()}</b>"
        lignes_html.append(ligne if ligne else "<div style='height:6px'></div>")
    vider_liste()
    sortie = ""
    for i, l in enumerate(lignes_html):
        bloc = l.startswith("<ul")
        precedent_bloc = i > 0 and lignes_html[i - 1].startswith("<ul")
        sortie += ("" if i == 0 or bloc or precedent_bloc else "<br>") + l
    return sortie

def df_vers_html(df):
    entetes = "".join(f"<th style='background:#0B2545;color:#fff;padding:5px 8px;text-align:left;font-weight:600'>{html.escape(str(c))}</th>" for c in df.columns)
    corps = ""
    for i, (_, ligne) in enumerate(df.iterrows()):
        fond = "#F4F7F5" if i % 2 else "#FFFFFF"
        corps += "<tr>" + "".join(f"<td style='padding:5px 8px;border-bottom:1px solid #DDE3E0;background:{fond};vertical-align:top'>"
                                  f"{html.escape(str(v))}</td>" for v in ligne.values) + "</tr>"
    return (f"<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:12px;width:100%;"
            f"font-family:Segoe UI,system-ui,sans-serif;color:#1d2b24'><tr>{entetes}</tr>{corps}</table></div>")

POLICE = "font-family:Segoe UI,system-ui,-apple-system,sans-serif"

def bulle_utilisateur(question):
    return (f"<div style='display:flex;justify-content:flex-end;margin:10px 0'>"
            f"<div style='max-width:78%;background:#0B2545;color:#fff;padding:10px 14px;border-radius:14px 14px 2px 14px;"
            f"{POLICE};font-size:14px;line-height:1.5'>🧑‍💼 {html.escape(question)}</div></div>")

def bulle_agent(texte, meta="", details_html="", couleur=VERT):
    details = (f"<details style='margin-top:8px'><summary style='cursor:pointer;color:#12507A;font-size:12.5px'>"
               f"🧭 Voir le raisonnement et les contrôles</summary><div style='margin-top:6px'>{details_html}</div></details>"
               if details_html else "")
    return (f"<div style='display:flex;justify-content:flex-start;margin:10px 0'>"
            f"<div style='max-width:86%;background:#FFFFFF;color:#1d2b24;border:1px solid #DDE3E0;border-left:5px solid {couleur};"
            f"padding:10px 14px;border-radius:14px 14px 14px 2px;{POLICE};font-size:14px;line-height:1.6;box-shadow:0 1px 3px rgba(0,0,0,.06)'>"
            f"<div>🤖 {texte_vers_html(texte)}</div>"
            f"<div style='margin-top:8px;color:#6B7B75;font-size:11.5px'>{meta}</div>{details}</div></div>")

def bulle_systeme(texte, couleur=OR):
    return (f"<div style='margin:8px auto;max-width:90%;text-align:center;background:#FFF7E0;border:1px dashed {couleur};"
            f"color:#5c4500;padding:6px 12px;border-radius:10px;{POLICE};font-size:12.5px'>{texte}</div>")


def traiter_question(question, cerveau, mode="agent", moteur="B · Amélioré", max_etapes=6, file=None):
    """Point d’entrée unique, partagé par le mode texte et l’interface."""
    file = file or FILE
    moteur_obj = MOTEURS[moteur]
    debut = time.perf_counter()
    appels_avant = cerveau.appels
    if mode == "rag":
        generateur = None if isinstance(cerveau, ModeleRegles) else cerveau
        r = repondre(question, moteur=moteur_obj, generateur=generateur)
        duree = time.perf_counter() - debut
        return {"question": question, "reponse": r["réponse"], "mode": "RAG simple", "execution": None,
                "ancrage": None, "ancre": None, "etapes": 0, "duree": duree, "appels": cerveau.appels - appels_avant,
                "details": f"<div style='{POLICE};font-size:12.5px'>Mode : <b>{r['mode']}</b> · confiance du meilleur passage : "
                           f"<b>{r['confiance']:.3f}</b> · règle : cosinus ≥ {SEUIL_COSINUS} ou couverture ≥ {SEUIL_COUVERTURE} · moteur : <b>{moteur}</b></div>"}
    outils = construire_outils(moteur_obj)
    file.tache_courante = question
    agent = Agent(outils, cerveau, politique=file.politique_file_approbation, max_etapes=max_etapes)
    run = agent.executer(question, bavard=False)
    rapport, ancre = verifier_ancrage(run)
    duree = time.perf_counter() - debut
    details = (f"<div style='{POLICE};font-size:12.5px;margin-bottom:4px'><b>Trace de l’agent</b> · politique : "
               f"<code>{run.politique}</code> · arrêt : {run.arret}</div>" + df_vers_html(run.journal().drop(columns=["ms"]))
               + f"<div style='{POLICE};font-size:12.5px;margin:8px 0 4px'><b>Contrôle d’ancrage</b></div>"
               + (df_vers_html(rapport) if not rapport.empty else "<i>aucun nombre ni source à vérifier</i>"))
    return {"question": question, "reponse": run.reponse, "mode": "Agent", "execution": run, "ancrage": rapport,
            "ancre": ancre, "etapes": len(run.etapes), "duree": duree, "appels": cerveau.appels - appels_avant, "details": details}


def meta_reponse(res, cerveau):
    badge = "" if res["ancre"] is None else (" · ✅ chiffres ancrés" if res["ancre"] else " · 🚩 à vérifier")
    return (f"{res['mode']} · {cerveau!r} · {res['etapes']} étape(s) · {res['appels']} appel(s) au modèle · "
            f"{res['duree']:.1f} s{badge}")


CERVEAU_DEMO = ModeleRegles()

def demander(question, mode="agent", moteur="B · Amélioré", cerveau=None, max_etapes=6):
    """Mode texte : même traitement et même rendu que l’interface."""
    cerveau = cerveau or CERVEAU_DEMO
    res = traiter_question(question, cerveau, mode, moteur, max_etapes)
    display(HTML(bulle_utilisateur(question) + bulle_agent(res["reponse"], meta_reponse(res, cerveau), res["details"],
                                                          couleur=VERT if res["ancre"] in (True, None) else ROUGE)))
    return res

_ = demander("Quel est le taux de chômage des jeunes, et quel écart avec le taux global ?")
_ = demander("Quel est le taux de change du dollar américain ?")

In [ ]:
# ── Étape 9 (suite) · Auto-test : la chaîne complète sur six demandes types ──
tests = [
    ("Quelle est l’inflation en mars 2024 ?", "4,1"),
    ("Quel est le taux de chomage des jeunes ?", "19,2"),
    ("De combien le chômage des jeunes dépasse-t-il le taux global ?", "10,6"),
    ("Combien de documents sont disponibles ?", "9"),
    ("Quel est le taux de change du dollar ?", "aucune information"),
    ("Prépare une note sur la production de cacao", "file d’approbation"),
]
lignes = []
file_test = FileApprobation(OUTILS_PAR_NOM)
for question, attendu in tests:
    res = traiter_question(question, ModeleRegles(), file=file_test)
    reussi = attendu.lower() in res["reponse"].lower()
    lignes.append({"demande": question, "attendu": attendu, "résultat": "✅" if reussi else "❌",
                   "étapes": res["etapes"], "ancrage": "✅" if res["ancre"] else ("—" if res["ancre"] is None else "🚩")})
bilan_tests = pd.DataFrame(lignes)
tableau(bilan_tests, f"{(bilan_tests['résultat'] == '✅').sum()}/{len(tests)} demandes traitées comme attendu. "
        f"Demandes d’écriture mises en attente pendant le test : {len(file_test.en_attente())} (aucun fichier écrit).")

---
# 🟥 PARTIE 5 — Utiliser l’assistant

## Étape 10 · L’interface de conversation

**🎯 Objectif :** mettre l’agent entre les mains d’un utilisateur, **sans rien perdre des contrôles**.

### 🧑‍🏫 Mode d’emploi

| Zone | Rôle |
|:--|:--|
| **⚙️ Réglages** | Choisir le cerveau, le mode (agent ou RAG simple), le moteur de recherche (A ou B) et le budget d’étapes |
| **💡 Suggestions** | Un clic remplit la question |
| **💬 Conversation** | Vos questions à droite, les réponses à gauche, avec leurs sources en pastilles 🔎 🧮 📚 📝 |
| **🧭 Trace** | Chaque étape de la dernière réponse : demande, décision, résultat |
| **✅ Approbations** | Les écritures en attente : **Approuver** ou **Rejeter** |
| **🧾 Journal** | Les journaux d’audit enregistrés, leur intégrité, et l’export de la conversation |
| **📊 Statistiques** | Questions, appels au modèle, refus, taux d’ancrage |

### 🔬 Expériences à faire

1. Posez **la même question** avec le moteur **A** puis **B** : observez la source retenue.
2. Demandez **« Prépare une note sur l’inflation »**, puis ouvrez l’onglet **✅ Approbations**.
3. Posez une question **hors corpus** (« Quel est le prix du pétrole ? ») : l’assistant doit refuser d’inventer.
4. Réglez le **budget** sur 2 étapes et demandez un écart : que se passe-t-il ?
5. Avec une clé Groq, comparez la **démo locale** et le **vrai modèle** : lequel respecte le mieux les règles ?

> ℹ️ **Si l’interface ne s’affiche pas** (aperçu GitHub, éditeur sans widgets), utilisez `demander("votre question")` de l’étape 9.

In [ ]:
# ── Étape 10 · L’interface ──────────────────────────────────────────────────
import warnings
import ipywidgets as widgets

CERVEAUX = [("🧪 Démo locale (sans clé)", "demo"), ("⚡ Groq — gratuit", "groq"),
            ("🔷 Gemini — gratuit", "gemini"), ("🖥️ Ollama — local", "ollama")]
SUGGESTIONS = [
    "Quel est le taux de chômage des jeunes ?",
    "De combien le chômage des jeunes dépasse-t-il le taux global ?",
    "Quelle est l’inflation en mars 2024 ?",
    "Combien de documents sont disponibles ?",
    "Prépare une note sur la production de cacao",
    "Quel est le prix du pétrole ?",
]


class InterfaceAssistant:
    def __init__(self, file):
        self.file = file
        self.fil = []                       # blocs HTML de la conversation
        self.resultats = []
        self.journaux = []
        self.cache_cerveaux = {"demo": ModeleRegles()}
        self.dossier = SORTIES / "interface"
        self.dossier.mkdir(parents=True, exist_ok=True)
        self._construire()
        self._accueil()

    # ── Construction ───────────────────────────────────────────────────────
    def _construire(self):
        L = widgets.Layout
        entete = widgets.HTML(
            f"<div style='background:linear-gradient(135deg,#0B2545 0%,#12507A 50%,#1B7A43 100%);border-radius:14px;"
            f"padding:14px 20px;{POLICE}'><div style='color:#F2A900;font-size:11px;letter-spacing:2.5px;font-weight:700'>"
            f"INSTITUT NATIONAL DE LA STATISTIQUE · {PAYS['nom'].upper()} · DONNÉES FICTIVES</div>"
            f"<div style='color:#fff;font-size:21px;font-weight:800;margin-top:3px'>🤖 Assistant statistique</div>"
            f"<div style='color:#d6e4ef;font-size:13px'>Posez une question sur les publications. Chaque chiffre est sourcé, "
            f"chaque action est contrôlée, chaque échange est journalisé.</div></div>")

        style = {"description_width": "90px"}
        self.w_cerveau = widgets.Dropdown(options=CERVEAUX, value="demo", description="🧠 Cerveau", style=style, layout=L(width="300px"))
        self.w_mode = widgets.Dropdown(options=[("🤖 Agent (outils)", "agent"), ("📄 RAG simple", "rag")], value="agent",
                                       description="🎛️ Mode", style=style, layout=L(width="260px"))
        self.w_moteur = widgets.Dropdown(options=list(MOTEURS), value="B · Amélioré", description="🔎 Moteur",
                                         style=style, layout=L(width="250px"))
        self.w_budget = widgets.IntSlider(value=6, min=2, max=10, description="⏱️ Budget", style=style, layout=L(width="300px"))
        reglages = widgets.VBox([widgets.HBox([self.w_cerveau, self.w_mode]), widgets.HBox([self.w_moteur, self.w_budget])])

        boutons = []
        for s in SUGGESTIONS:
            b = widgets.Button(description=s if len(s) <= 46 else s[:44] + "…", tooltip=s,
                               layout=L(width="auto", margin="2px"))
            b.on_click(lambda _, q=s: self._suggerer(q))
            boutons.append(b)
        suggestions = widgets.HBox(boutons, layout=L(flex_flow="row wrap"))

        self.w_conv = widgets.HTML(layout=L(width="100%"))
        self.w_question = widgets.Text(placeholder="Écrivez votre question puis appuyez sur Entrée…", layout=L(width="70%"))
        self.w_envoyer = widgets.Button(description="Envoyer", icon="paper-plane", button_style="success", layout=L(width="130px"))
        self.w_effacer = widgets.Button(description="Effacer", icon="trash", layout=L(width="110px"))
        self.w_envoyer.on_click(self._envoyer)
        self.w_effacer.on_click(self._effacer)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                self.w_question.on_submit(self._envoyer)
            except Exception:
                pass
        saisie = widgets.HBox([self.w_question, self.w_envoyer, self.w_effacer])
        self.w_statut = widgets.HTML()

        self.w_trace = widgets.HTML()
        self.w_approbations = widgets.VBox()
        self.w_journal = widgets.HTML()
        self.w_exporter = widgets.Button(description="Exporter la conversation", icon="download", layout=L(width="240px"))
        self.w_exporter.on_click(self._exporter)
        self.w_export_msg = widgets.HTML()
        self.w_stats = widgets.HTML()
        self.onglets = widgets.Tab(children=[self.w_trace, self.w_approbations,
                                             widgets.VBox([self.w_journal, widgets.HBox([self.w_exporter, self.w_export_msg])]),
                                             self.w_stats])
        for i, titre in enumerate(["🧭 Trace", "✅ Approbations", "🧾 Journal", "📊 Statistiques"]):
            self.onglets.set_title(i, titre)

        def titre_zone(t):
            return widgets.HTML(f"<div style='{POLICE};font-weight:700;color:#0B2545;margin:10px 0 2px'>{t}</div>")

        self.app = widgets.VBox([entete, titre_zone("⚙️ Réglages"), reglages, titre_zone("💡 Suggestions"), suggestions,
                                 titre_zone("💬 Conversation"), self.w_conv, saisie, self.w_statut,
                                 titre_zone("🔬 Contrôles et traçabilité"), self.onglets],
                                layout=L(width="100%", max_width="1000px"))

    # ── Affichage ──────────────────────────────────────────────────────────
    def _accueil(self):
        self.fil = [bulle_agent("Bonjour ! Je réponds à partir des **publications de l’institut** (données fictives).\n"
                                "- Je **cite** la source de chaque chiffre ;\n"
                                "- je **calcule** avec un outil, sans estimer ;\n"
                                "- toute **écriture** attend la validation d’un analyste.\n"
                                "Choisissez une suggestion ou écrivez votre question.", "Assistant prêt")]
        self._rafraichir()

    def _rafraichir(self):
        self.w_conv.value = (f"<div style='background:#F7FAF8;border:1px solid #DDE3E0;border-radius:12px;padding:6px 12px;"
                             f"height:430px;overflow-y:auto;{POLICE}'>" + "".join(self.fil) + "</div>")
        self._rafraichir_approbations()
        self._rafraichir_journal()
        self._rafraichir_stats()

    def _statut(self, texte, couleur="#6B7B75"):
        self.w_statut.value = f"<div style='{POLICE};font-size:12.5px;color:{couleur};margin:4px 2px'>{texte}</div>"

    def _rafraichir_approbations(self):
        attente = self.file.en_attente()
        blocs = [widgets.HTML(f"<div style='{POLICE};font-size:13px;margin:6px 0'>"
                              f"<b>{len(attente)}</b> demande(s) en attente · {len(self.file.decisions)} décision(s) prise(s)</div>")]
        for d in attente:
            apercu = html.escape(str(d.args.get("texte", ""))[:220]).replace("\n", "<br>")
            carte = widgets.HTML(
                f"<div style='{POLICE};border:1px solid #F2A900;background:#FFFBEF;border-radius:10px;padding:8px 12px;font-size:12.5px'>"
                f"<b>Demande n°{d.numero}</b> · <code>{d.outil}</code> → <code>{html.escape(str(d.args.get('nom_fichier', '')))}</code>"
                f"<br><span style='color:#6B7B75'>Question d’origine : {html.escape(d.tache)}</span>"
                f"<div style='margin-top:6px;background:#fff;border:1px solid #EEE;border-radius:6px;padding:6px'>{apercu}</div></div>",
                layout=widgets.Layout(width="68%"))
            ok = widgets.Button(description="Approuver", icon="check", button_style="success", layout=widgets.Layout(width="120px"))
            ko = widgets.Button(description="Rejeter", icon="times", button_style="danger", layout=widgets.Layout(width="110px"))
            ok.on_click(lambda _, n=d.numero: self._decider(n, True))
            ko.on_click(lambda _, n=d.numero: self._decider(n, False))
            blocs.append(widgets.HBox([carte, widgets.VBox([ok, ko])], layout=widgets.Layout(margin="4px 0")))
        if self.file.decisions:
            historique = pd.DataFrame([{"n°": e["numero"], "outil": e["outil"], "décision": e["statut"],
                                        "par": e["decideur"], "à (UTC)": e["horodatage"], "empreinte": e["hash"][:12] + "…"}
                                       for e in self.file.decisions])
            blocs.append(widgets.HTML(f"<div style='{POLICE};font-size:12.5px;margin:8px 0 4px'><b>Historique chaîné des décisions</b></div>"
                                      + df_vers_html(historique)))
        self.w_approbations.children = blocs

    def _rafraichir_journal(self):
        if not self.journaux:
            self.w_journal.value = f"<div style='{POLICE};font-size:13px;color:#6B7B75'>Aucun journal pour l’instant.</div>"
            return
        lignes = []
        for chemin, question in self.journaux[-12:]:
            ok, etat = verifier_journal(chemin)
            lignes.append({"fichier": chemin.name, "question": question[:60], "intégrité": f"{'🔒' if ok else '🚩'} {etat}"})
        self.w_journal.value = (f"<div style='{POLICE};font-size:12.5px;margin:6px 0'>Dossier : <code>{self.dossier}</code></div>"
                                + df_vers_html(pd.DataFrame(lignes)))

    def _rafraichir_stats(self):
        agents = [r for r in self.resultats if r["mode"] == "Agent"]
        refus = sum(1 for r in agents for e in r["execution"].etapes if e.autorise is False)
        ancres = sum(1 for r in agents if r["ancre"])
        cartes = [("Questions", len(self.resultats)), ("Appels au modèle", sum(r["appels"] for r in self.resultats)),
                  ("Actions refusées ou en attente", refus), ("Réponses ancrées", f"{ancres}/{len(agents)}" if agents else "—"),
                  ("Écritures approuvées", sum(1 for d in self.file.demandes if d.statut == "approuvée"))]
        self.w_stats.value = ("<div style='display:flex;flex-wrap:wrap;gap:10px;margin:8px 0'>" + "".join(
            f"<div style='{POLICE};background:#fff;border:1px solid #DDE3E0;border-top:4px solid {VERT};border-radius:10px;"
            f"padding:10px 16px;min-width:150px'><div style='font-size:24px;font-weight:800;color:#0B2545'>{v}</div>"
            f"<div style='font-size:12px;color:#6B7B75'>{t}</div></div>" for t, v in cartes) + "</div>")

    # ── Actions ────────────────────────────────────────────────────────────
    def _cerveau(self):
        cle = self.w_cerveau.value
        if cle not in self.cache_cerveaux:
            self.cache_cerveaux[cle] = ModeleCompatibleOpenAI(cle)
        return self.cache_cerveaux[cle]

    def _suggerer(self, question):
        self.w_question.value = question

    def _envoyer(self, _=None):
        question = self.w_question.value.strip()
        if not question:
            self._statut("✍️ Écrivez d’abord une question.", OR)
            return
        self.w_question.value = ""
        self.w_envoyer.disabled = True
        self.fil.append(bulle_utilisateur(question))
        self._rafraichir()
        self._statut("⏳ L’assistant réfléchit…", "#12507A")
        try:
            cerveau = self._cerveau()
            res = traiter_question(question, cerveau, self.w_mode.value, self.w_moteur.value, self.w_budget.value, self.file)
            self.resultats.append(res)
            couleur = VERT if res["ancre"] in (True, None) else ROUGE
            self.fil.append(bulle_agent(res["reponse"], meta_reponse(res, cerveau), res["details"], couleur))
            if res["execution"] is not None:
                chemin = self.dossier / f"journal_{len(self.journaux) + 1:03d}.json"
                sauvegarder_journal(res["execution"], chemin, {"pays": PAYS["iso3"], "cerveau": repr(cerveau),
                                                               "moteur": self.w_moteur.value, "budget": self.w_budget.value})
                self.journaux.append((chemin, question))
                self.w_trace.value = res["details"]
                attente = [e for e in res["execution"].etapes if e.motif.startswith("mise en file")]
                if attente:
                    self.fil.append(bulle_systeme("🕒 Une écriture attend votre décision dans l’onglet <b>✅ Approbations</b>."))
                    self.onglets.selected_index = 1
            else:
                self.w_trace.value = res["details"]
            self._statut(f"✅ Réponse en {res['duree']:.1f} s · {cerveau!r}", VERT)
        except Exception as exc:
            self.fil.append(bulle_agent(f"Je n’ai pas pu traiter la demande : **{type(exc).__name__}** — {exc}\n"
                                        "Vérifiez la clé du cerveau choisi, ou revenez à la **démo locale**.", "Erreur", couleur=ROUGE))
            self._statut(f"⚠️ {type(exc).__name__} : {exc}", ROUGE)
        finally:
            self.w_envoyer.disabled = False
            self._rafraichir()

    def _decider(self, numero, approuver):
        d = self.file.decider(numero, approuver)
        if approuver:
            self.fil.append(bulle_systeme(f"✅ Demande n°{numero} <b>approuvée</b> par l’analyste — {html.escape(d.resultat)}", VERT))
        else:
            self.fil.append(bulle_systeme(f"⛔ Demande n°{numero} <b>rejetée</b> par l’analyste — aucun fichier écrit.", ROUGE))
        self._rafraichir()

    def _effacer(self, _=None):
        self.resultats.clear()
        self._accueil()
        self._statut("🧹 Conversation effacée (les journaux d’audit sont conservés).")

    def _exporter(self, _=None):
        horodatage = datetime.now().strftime("%Y%m%d_%H%M%S")
        chemin = self.dossier / f"conversation_{horodatage}.md"
        lignes = [f"# Conversation — assistant statistique ({PAYS['nom']}, données fictives)", ""]
        for r in self.resultats:
            lignes += [f"## 🧑‍💼 {r['question']}", "", r["reponse"], "",
                       f"*{r['mode']} · {r['etapes']} étape(s) · ancrage : "
                       f"{'—' if r['ancre'] is None else ('complet' if r['ancre'] else 'à vérifier')}*", ""]
        chemin.write_text("\n".join(lignes), encoding="utf-8")
        self.w_export_msg.value = f"<span style='{POLICE};font-size:12.5px'>💾 <code>{chemin}</code></span>"
        try:
            from google.colab import files
            files.download(str(chemin))
        except Exception:
            pass

    def afficher(self):
        display(self.app)


APP = InterfaceAssistant(FILE)
APP.afficher()

> 💡 **Astuce :** l’interface reste active tant que la session tourne. Vous pouvez aussi la piloter par le code, par exemple `APP.w_question.value = "..."` puis `APP._envoyer()`.

## Étape 11 · Sous le capot

**🎯 Objectif :** savoir expliquer à votre direction ce qui se passe quand un utilisateur clique sur **Envoyer**.

**🔍 À observer dans le schéma :** l’interface ne parle **jamais** directement aux outils. Tout passe par la boucle de l’agent et par la politique ; toute écriture passe par un humain ; tout est journalisé.

In [ ]:
# ── Étape 11 · Architecture de l’assistant ──────────────────────────────────
def _boite(x, y, w, h, titre, sous, bord, fond="#FFFFFF"):
    return (f"<rect x='{x}' y='{y}' width='{w}' height='{h}' rx='12' fill='{fond}' stroke='{bord}' stroke-width='2'/>"
            f"<text x='{x + w / 2}' y='{y + h / 2 - 4}' text-anchor='middle' font-size='13.5' font-weight='700' fill='#1d2b24'>{titre}</text>"
            f"<text x='{x + w / 2}' y='{y + h / 2 + 14}' text-anchor='middle' font-size='11' fill='#56655e'>{sous}</text>")

def _fleche(x1, y1, x2, y2, coul="#56655e", texte="", dy=-6):
    return (f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}' stroke='{coul}' stroke-width='2' marker-end='url(#fl)'/>"
            + (f"<text x='{(x1 + x2) / 2}' y='{(y1 + y2) / 2 + dy}' text-anchor='middle' font-size='10.5' fill='{coul}' font-weight='600'>{texte}</text>" if texte else ""))

svg = f"""<svg viewBox='0 0 940 400' width='100%' style='max-width:940px;background:#fff;{POLICE}'>
<defs><marker id='fl' markerUnits='userSpaceOnUse' markerWidth='10' markerHeight='10' refX='9' refY='5' orient='auto'>
<path d='M0,0 L10,5 L0,10 z' fill='#56655e'/></marker></defs>
{_boite(20, 40, 150, 64, "🧑‍💼 Utilisateur", "pose une question", "#0B2545", "#E8EEF6")}
{_boite(20, 160, 150, 64, "🖥️ Interface", "ipywidgets", "#0B2545", "#E8EEF6")}
{_boite(240, 160, 170, 64, "🔁 Boucle de l’agent", "analyse · décide · exécute", VERT, CLAIR)}
{_boite(240, 30, 170, 64, "🧠 Cerveau", "démo · Groq · Gemini · Ollama", "#12507A", "#E8EEF6")}
{_boite(480, 160, 150, 64, "🛡️ Politique", "motif journalisé", OR, "#FFF7E0")}
{_boite(700, 90, 210, 64, "🔎 🧮 📚 Outils de lecture", "exécutés immédiatement", VERT, CLAIR)}
{_boite(700, 230, 210, 64, "🕒 File d’approbation", "écritures en attente", OR, "#FFF7E0")}
{_boite(480, 310, 150, 64, "✅ Ancrage", "chiffres vérifiés", VERT)}
{_boite(240, 310, 170, 64, "🧾 Journal chaîné", "SHA-256 · JSON", MARINE)}
{_boite(20, 310, 150, 64, "👩‍💼 Analyste", "approuve / rejette", "#0B2545", "#E8EEF6")}
{_fleche(95, 104, 95, 158, "#0B2545", "question", 0)}
{_fleche(170, 192, 238, 192, "#0B2545")}
{_fleche(325, 158, 325, 96, "#12507A", "consigne + historique", 0)}
{_fleche(410, 192, 478, 192, VERT, "demande")}
{_fleche(630, 180, 698, 128, VERT, "autorisé", -2)}
{_fleche(630, 204, 698, 256, OR, "écriture", 14)}
{_fleche(325, 226, 325, 308, MARINE, "chaque étape", 0)}
{_fleche(410, 226, 478, 318, VERT, "réponse")}
{_fleche(700, 280, 172, 338, OR, "à décider", -8)}
</svg>"""
display(HTML(svg))

tableau(pd.DataFrame([
    {"n°": 1, "où": "Interface", "ce qui se passe": "la question est lue, les réglages sont appliqués (cerveau, mode, moteur, budget)"},
    {"n°": 2, "où": "Boucle", "ce qui se passe": "le cerveau reçoit la consigne et propose une action en JSON"},
    {"n°": 3, "où": "Analyseur", "ce qui se passe": "JSON illisible ou outil inconnu → l’erreur est renvoyée au cerveau"},
    {"n°": 4, "où": "Politique", "ce qui se passe": "lecture → exécutée ; donnée individuelle → refusée ; écriture conforme → file d’approbation"},
    {"n°": 5, "où": "Outils", "ce qui se passe": "le résultat est renvoyé au cerveau, qui décide de la suite"},
    {"n°": 6, "où": "Budget", "ce qui se passe": "au-delà de max_etapes, la boucle s’arrête, quoi qu’il arrive"},
    {"n°": 7, "où": "Ancrage", "ce qui se passe": "chaque nombre de la réponse est cherché dans les résultats d’outils"},
    {"n°": 8, "où": "Journal", "ce qui se passe": "les réponses brutes et les décisions sont enregistrées et chaînées"},
    {"n°": 9, "où": "Analyste", "ce qui se passe": "il approuve ou rejette les écritures ; sa décision est chaînée à son tour"},
]), "Le trajet complet d’une question.")

---
# ⬛ PARTIE 6 — Conclure

## Étape 12 · Limites, exercices et dépannage

### ⚠️ Les limites de ce que vous avez construit

- **Le cerveau de démonstration n’est pas intelligent.** Il applique des règles fixes ; il sert à exercer la chaîne de contrôle, pas à juger la qualité des réponses.
- **Chaque question est traitée indépendamment.** L’interface affiche l’historique, mais l’agent ne s’en sert pas : « et pour les femmes ? » ne sera pas compris.
- **Le contrôle d’ancrage vérifie les nombres, pas le raisonnement.** Un chiffre juste peut être attribué au mauvais indicateur.
- **Le protocole est textuel.** L’appel d’outils natif des fournisseurs serait plus robuste en production.
- **L’interface tourne dans un notebook.** Pour vos collègues, il faudra une application web avec authentification (Streamlit, Gradio, Dash…).

### ✅ Point de contrôle

| Question | Réponse |
|:--|:--|
| Pourquoi comparer deux moteurs de recherche ? | Parce que la qualité de la réponse dépend d’abord du passage retrouvé |
| Pourquoi une **file** d’approbation dans une interface ? | On ne peut pas bloquer l’agent en attendant un humain : l’écriture attend, l’agent répond |
| Qu’est-ce qui prouve qu’une écriture n’a pas eu lieu ? | **Le disque**, pas seulement le journal |
| Que détecte le contrôle d’ancrage ? | Les chiffres absents des résultats d’outils et les sources jamais appelées |
| Pourquoi le cerveau est-il interchangeable ? | Parce que tout le contrôle est dans **votre** code, pas dans le modèle |

### 🧑‍💻 À vous de jouer

1. **Mémoire de conversation.** Modifiez `traiter_question` pour transmettre les deux derniers échanges au cerveau. Testez « et pour les femmes ? ».
2. **Nouvel outil.** Ajoutez `comparer_periodes(indicateur)` et justifiez en deux phrases ce qu’une demande malveillante pourrait en faire.
3. **Rôles.** Ajoutez un champ « analyste » à l’interface et interdisez qu’une même personne pose la question **et** approuve l’écriture.
4. **Qualité.** Ajoutez un bouton 👍 / 👎 sous chaque réponse et enregistrez les avis dans le journal.
5. **Production.** Transposez `InterfaceAssistant` en application Gradio ou Streamlit.

### 🛟 Dépannage

| Symptôme | Solution |
|:--|:--|
| L’interface ne s’affiche pas | Réexécutez la cellule de l’étape 10 ; sinon utilisez `demander("…")` |
| « aucun fournisseur » ou « clé absente » | Ajoutez `GROQ_API_KEY` dans 🔑 Secrets et autorisez le notebook, ou restez en démo locale |
| Erreur 429 répétée | Quota gratuit atteint : patientez, ou changez de cerveau |
| Erreur 404 « model not found » | Les noms de modèles évoluent : mettez à jour `FOURNISSEURS[...]["modele"]` à l’étape 8 |
| Le vrai modèle invente un chiffre | C’est un constat utile : la bulle devient rouge et l’onglet 🧭 Trace montre l’ancrage 🚩 |
| Repartir de zéro | Menu **Exécution → Redémarrer et tout exécuter** |

---

*Atelier STG17 · BAD / STATAFRIC · « Assistant statistique agentique » · Toutes les données sont fictives.*